# RWA.watch V6 Analytics Builder (Colab-ready)

This notebook builds the **separate Analytics site** for RWA.watch.

What it does:

- fetches the current RWA asset universe from DefiLlama structured APIs (same ingest style as the main build)
- normalizes and validates the dataset
- creates an **analytics-only snapshot** (separate namespace from the main site)
- maintains internal CSV exports and validation files
- renders daily and weekly analytics HTML pages
- renders PNG charts in the blue tech style
- packages everything into `rwa_analysis_site.zip`

Hard rules baked in:

- **no writes to the main-site data paths**
- **analytics snapshot lives in a separate namespace**
- **facts only / no ratings**
- **English public copy**
- **UTC-based dates**
- **ID tracking by slug**



In [ ]:
# Colab package bootstrap
!pip -q install requests pandas numpy matplotlib beautifulsoup4 lxml python-dateutil cloudscraper


In [ ]:
from __future__ import annotations

import csv
import html
import io
import json
import hashlib
import math
import os
import random
import re
import shutil
import subprocess
import textwrap
import zipfile
from collections import defaultdict
from copy import deepcopy
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone, date
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple
from urllib.parse import urljoin, urlparse

import numpy as np
import pandas as pd
import requests

try:
    import cloudscraper
except Exception:
    cloudscraper = None
from bs4 import BeautifulSoup
from dateutil import parser as dtparser
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

UTC = timezone.utc
TODAY_UTC = datetime.now(UTC).date()
RUN_TS_UTC = datetime.now(UTC)

# ----------------------------
# USER CONFIG
# ----------------------------

# Optional: mount Google Drive first and point this to a persistent folder if you want history to survive across Colab sessions.
# Example: ROOT_DIR = Path('/content/drive/MyDrive/rwa_watch_analytics')
# Build output root.
# - In Colab, /content is common.
# - In a local repo / GitHub Actions, fall back to the current working directory.
_DEFAULT_ROOT = Path("/content/rwa_watch_analytics")
ROOT_DIR = _DEFAULT_ROOT if _DEFAULT_ROOT.parent.exists() else (Path.cwd() / "rwa_watch_analytics")

# Optional manual snapshot date override in UTC (YYYY-MM-DD). Leave as None to use today's UTC date.
SNAPSHOT_DATE_OVERRIDE = None

# Main site shell source. If present, the notebook will try to read it for reference.
# The notebook still works without this because it embeds the shared shell internally.
MAIN_INDEX_HTML_PATH = Path('/content/index.html')

# Optional DefiLlama pro API key. Leave empty for public/free mode.
LLAMA_API_KEY = os.environ.get('LLAMA_API_KEY', '').strip()

# Whether to keep non-public investor CSVs inside the ZIP package.
# These files are generated regardless; this flag only controls whether they are included in the final zip.
INCLUDE_PRIVATE_EXPORTS_IN_ZIP = True

# Asset identity registry (optional). If present, we use canonical_id from this registry as the stable asset_id.
ASSET_REGISTRY_PATH = Path('generator/identity/asset_registry.json')
# If True, automatically update first_seen/last_seen and add new assets into the registry file.
AUTO_UPDATE_ASSET_REGISTRY = True


# ----------------------------
# PATHS / NAMESPACES
# ----------------------------

SNAPSHOT_DATE = date.fromisoformat(SNAPSHOT_DATE_OVERRIDE) if SNAPSHOT_DATE_OVERRIDE else TODAY_UTC
SNAPSHOT_DATE_STR = SNAPSHOT_DATE.isoformat()
ISO_YEAR, ISO_WEEK, ISO_WEEKDAY = SNAPSHOT_DATE.isocalendar()
WEEK_LABEL = f"{ISO_YEAR}-W{ISO_WEEK:02d}"

ANALYTICS_ROOT = ROOT_DIR / "analytics_build"
PRIVATE_DATA_ROOT = ANALYTICS_ROOT / "_private"
SNAPSHOT_ROOT = PRIVATE_DATA_ROOT / "snapshots"
NORMALIZED_ROOT = PRIVATE_DATA_ROOT / "normalized"
VALIDATION_ROOT = PRIVATE_DATA_ROOT / "validation"
ARTICLE_ROOT = PRIVATE_DATA_ROOT / "articles"
CHARTS_ROOT = PRIVATE_DATA_ROOT / "charts"
EXPORTS_ROOT = PRIVATE_DATA_ROOT / "exports" / "csv"

SITE_ROOT = ANALYTICS_ROOT / "site"
SITE_ANALYSIS_ROOT = SITE_ROOT / "analysis"
SITE_WEEKLY_ROOT = SITE_ANALYSIS_ROOT / "weekly"
SITE_CHARTS_ROOT = SITE_ANALYSIS_ROOT / "charts"
SITE_DATA_ROOT = SITE_ANALYSIS_ROOT / "data"
SITE_DATA_DAILY_ROOT = SITE_DATA_ROOT / "daily"
SITE_DATA_WEEKLY_ROOT = SITE_DATA_ROOT / "weekly"
SITE_DATA_METRICS_ROOT = SITE_DATA_ROOT / "metrics"
SITE_VALIDATION_ROOT = SITE_ANALYSIS_ROOT / "validation"
SITE_EXPORTS_ROOT = SITE_ANALYSIS_ROOT / "exports" / "csv"

ZIP_PATH = ROOT_DIR / "rwa_analysis_site.zip"

ALL_DIRS = [
    ROOT_DIR,
    ANALYTICS_ROOT,
    PRIVATE_DATA_ROOT,
    SNAPSHOT_ROOT,
    NORMALIZED_ROOT,
    VALIDATION_ROOT,
    ARTICLE_ROOT,
    CHARTS_ROOT,
    EXPORTS_ROOT,
    SITE_ROOT,
    SITE_ANALYSIS_ROOT,
    SITE_WEEKLY_ROOT,
    SITE_CHARTS_ROOT,
    SITE_DATA_ROOT,
    SITE_DATA_DAILY_ROOT,
    SITE_DATA_WEEKLY_ROOT,
    SITE_DATA_METRICS_ROOT,
    SITE_VALIDATION_ROOT,
    SITE_EXPORTS_ROOT,
]

for p in ALL_DIRS:
    p.mkdir(parents=True, exist_ok=True)

# Fail-fast safety guards so the analytics builder cannot write into the main-site namespace by accident.
FORBIDDEN_SEGMENTS = [
    "/site/data/",
    "/data/snapshots/",
]

allowed_prefixes = [
    str(ANALYTICS_ROOT.resolve()),
    str(ROOT_DIR.resolve()),
]

for candidate in [SNAPSHOT_ROOT, NORMALIZED_ROOT, VALIDATION_ROOT, EXPORTS_ROOT, SITE_ANALYSIS_ROOT]:
    resolved = str(candidate.resolve())
    if not any(resolved.startswith(prefix) for prefix in allowed_prefixes):
        raise RuntimeError(f"Unsafe analytics output path: {resolved}")

    if any(seg in resolved.replace('\\', '/') for seg in FORBIDDEN_SEGMENTS):
        # Allow /site/analysis/data/*, but never /site/data/*
        if "/site/analysis/data/" not in resolved.replace('\\', '/'):
            raise RuntimeError(f"Blocked forbidden path segment in analytics output path: {resolved}")

print("ROOT_DIR:", ROOT_DIR)
print("SNAPSHOT_DATE:", SNAPSHOT_DATE_STR)
print("WEEK_LABEL:", WEEK_LABEL)


# ----------------------------
# FILE / TIME HELPERS
# ----------------------------

try:
    from IPython.display import display
except Exception:
    def display(*args, **kwargs):
        for arg in args:
            print(arg)


def iso_ts(dt: datetime) -> str:
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=UTC)
    return dt.astimezone(UTC).replace(microsecond=0).isoformat().replace('+00:00', 'Z')


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def atomic_write_bytes(path: Path, data: bytes) -> None:
    """Write bytes atomically (temp file -> rename)."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(data)
    tmp.replace(path)

def atomic_write_text(path: Path, text: str, encoding: str = "utf-8") -> None:
    atomic_write_bytes(path, text.encode(encoding))

def write_json(path: Path, payload: Any) -> str:
    """Write JSON atomically and return sha256."""
    raw = json.dumps(payload, ensure_ascii=False, indent=2).encode("utf-8")
    atomic_write_bytes(Path(path), raw)
    return sha256_bytes(raw)

def write_text(path: Path, text: str, encoding: str = "utf-8") -> str:
    """Write text atomically and return sha256."""
    raw = text.encode(encoding)
    atomic_write_bytes(Path(path), raw)
    return sha256_bytes(raw)

def unique_path_if_exists(path: Path) -> Path:
    """If path exists, append __HHMMSS (UTC) before suffix."""
    path = Path(path)
    if not path.exists():
        return path
    ts = datetime.now(UTC).strftime("%H%M%S")
    return path.with_name(f"{path.stem}__{ts}{path.suffix}")

def save_source_archives(snapshot_id: str, protocols_raw: Any, yield_pools_raw: Any) -> Dict[str, Any]:
    """Persist raw source payloads alongside the snapshot for auditability."""
    # Keep sources under _private/sources/<snapshot_id>/
    sources_root = PRIVATE_DATA_ROOT / "sources" / snapshot_id
    sources_root.mkdir(parents=True, exist_ok=True)

    out = {}
    for name, payload in (("protocols.json", protocols_raw), ("yield_pools.json", yield_pools_raw)):
        p = sources_root / name
        sha = write_json(p, payload)
        out[name] = {"path": str(p.relative_to(ROOT_DIR)), "sha256": sha}
    return out


def read_json(path: Path) -> Any:
    return json.loads(Path(path).read_text(encoding='utf-8'))


def date_to_human(d: date) -> str:
    return d.strftime('%B %d, %Y')

# Public analysis base URL used for canonical links and sitemap generation.
PUBLIC_BASE_URL = "https://rwa.watch/analysis"


In [ ]:
# ----------------------------
# NETWORK / FETCH HELPERS
# ----------------------------

COMMON_HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0 Safari/537.36",
    "Accept": "text/html,application/json,text/csv,*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Cache-Control": "no-cache",
    "Pragma": "no-cache",
    "Referer": "https://defillama.com/",
    "Origin": "https://defillama.com",
}

def make_session() -> requests.Session:
    s = requests.Session()
    s.headers.update(COMMON_HEADERS)
    return s

SESSION = make_session()
SCRAPER = None
if cloudscraper is not None:
    try:
        SCRAPER = cloudscraper.create_scraper(browser={"browser": "chrome", "platform": "linux", "mobile": False})
        SCRAPER.headers.update(COMMON_HEADERS)
    except Exception:
        SCRAPER = None

def free_api_url(path: str) -> str:
    path = path if path.startswith("/") else "/" + path
    return "https://api.llama.fi" + path

def pro_api_url(path: str) -> str:
    path = path if path.startswith("/") else "/" + path
    return f"https://pro-api.llama.fi/{LLAMA_API_KEY}" + path

def _curl_get(url: str, timeout: int = 45) -> requests.Response:
    header_args = []
    for k, v in COMMON_HEADERS.items():
        header_args.extend(["-H", f"{k}: {v}"])
    cmd = [
        "curl", "-L", "--compressed", "--max-time", str(timeout), "--fail",
        *header_args,
        url,
    ]
    result = subprocess.run(cmd, capture_output=True, text=False)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.decode("utf-8", errors="ignore")[:400])
    resp = requests.Response()
    resp.status_code = 200
    resp._content = result.stdout
    resp.url = url
    return resp

def get_url(url: str, timeout: int = 45) -> requests.Response:
    errors = []
    for fetcher_name, fetcher in [
        ("cloudscraper", SCRAPER.get if SCRAPER is not None else None),
        ("requests", SESSION.get),
    ]:
        if fetcher is None:
            continue
        try:
            r = fetcher(url, timeout=timeout)
            r.raise_for_status()
            return r
        except Exception as e:
            errors.append(f"{fetcher_name}: {type(e).__name__}: {e}")
    try:
        return _curl_get(url, timeout=timeout)
    except Exception as e:
        errors.append(f"curl: {type(e).__name__}: {e}")
    raise RuntimeError(" | ".join(errors))

def get_text(url: str, timeout: int = 45) -> str:
    return get_url(url, timeout=timeout).text

def get_json(url: str, timeout: int = 45) -> Any:
    return get_url(url, timeout=timeout).json()

def maybe_get_json(url: str, timeout: int = 45) -> Optional[Any]:
    try:
        return get_json(url, timeout=timeout)
    except Exception:
        return None

def maybe_read_html_source(html_or_url: str) -> List[pd.DataFrame]:
    try:
        if "<html" in html_or_url.lower() or "<table" in html_or_url.lower():
            return pd.read_html(io.StringIO(html_or_url))
        return pd.read_html(html_or_url)
    except Exception:
        return []

def safe_float(v: Any) -> Optional[float]:
    if v is None:
        return None
    if isinstance(v, (int, float, np.integer, np.floating)):
        if pd.isna(v):
            return None
        return float(v)
    s = str(v).strip()
    if s == "" or s.lower() in {"nan", "none", "null", "-", "—", "n/a"}:
        return None
    s = s.replace("$", "").replace(",", "").replace("%", "")
    s = s.replace("−", "-")
    multiplier = 1.0
    if s.endswith("T"):
        multiplier = 1e12
        s = s[:-1]
    elif s.endswith("B"):
        multiplier = 1e9
        s = s[:-1]
    elif s.endswith("M"):
        multiplier = 1e6
        s = s[:-1]
    elif s.endswith("K"):
        multiplier = 1e3
        s = s[:-1]
    try:
        return float(s) * multiplier
    except Exception:
        return None

def slugify(text: str) -> str:
    text = (text or "").strip().lower()
    text = re.sub(r"&", " and ", text)
    text = re.sub(r"[^a-z0-9]+", "-", text)
    text = re.sub(r"-{2,}", "-", text).strip("-")
    return text or "unknown-asset"

def fmt_money_compact_abs(x: Optional[float]) -> str:
    if x is None or not np.isfinite(x):
        return "N/A"
    sign = "-" if x < 0 else ""
    x = abs(float(x))
    if x >= 1e12:
        return f"{sign}${x/1e12:.2f}T"
    if x >= 1e9:
        return f"{sign}${x/1e9:.2f}B"
    if x >= 1e6:
        return f"{sign}${x/1e6:.2f}M"
    if x >= 1e3:
        return f"{sign}${x/1e3:.2f}K"
    return f"{sign}${x:,.0f}"

def fmt_pct(x: Optional[float], digits: int = 1) -> str:
    if x is None or not np.isfinite(x):
        return "N/A"
    return f"{x:+.{digits}f}%"


def fmt_pct_plain(x: Optional[float], digits: int = 1) -> str:
    """Percent formatter without sign (for shares/concentration)."""
    if x is None or not np.isfinite(x):
        return "N/A"
    return f"{x:.{digits}f}%"
def fmt_money_compact_signed(x: Optional[float]) -> str:
    if x is None or not np.isfinite(x):
        return "N/A"
    sign = "-" if x < 0 else "+"
    return f"{sign}{fmt_money_compact_abs(abs(float(x)))}"



In [ ]:
# ----------------------------
# DEFILLAMA RWA CURRENT DATA INGEST
# ----------------------------

# This follows the main-site generator strategy:
# - fetch the structured DeFiLlama protocol dataset
# - fetch yield pools for auxiliary integrity flags
# - filter category == "RWA"
# - build an analytics-only dataset / snapshot in a separate namespace

DEFI_LLAMA_PROTOCOLS_URL = free_api_url("/protocols")
DEFI_LLAMA_YIELDS_POOLS_URL = "https://yields.llama.fi/pools"
RWA_PUBLIC_PAGE_URL = "https://defillama.com/rwa"

@dataclass
class YieldIntegrity:
    confidence: str
    sum_pool_tvl: float
    pool_count_used: int
    reported_high_apy_detected: bool
    reported_strong_apy_detected: bool

POOL_TVL_CUTOFF_USD = 100_000.0
APY_EXCLUDE_GT = 200.0
APY_STRONG_WARNING_GT = 100.0
APY_WARNING_GT = 50.0
LOW_COVERAGE_THRESHOLD_PCT = 25.0

PROTOCOL_CATEGORY_MAP = {
    "treasury": "Treasury",
    "treasury (tbills, bonds, mmfs)": "Treasury",
    "sovereign bond fund": "Treasury",
    "money market fund": "Treasury",
    "stablecoins": "Stablecoins",
    "private credit": "Private Credit",
    "real estate": "Real Estate",
    "commodities": "Commodities",
}

KNOWN_RWA_ASSET_NAME_TO_ISSUER = {
    "blackrock buidl": "BlackRock",
    "superstate ustb": "Superstate",
    "ondo global markets": "Ondo",
    "wisdomtree": "WisdomTree",
    "ethena usdtb": "Ethena",
    "tether gold": "Tether",
    "paxos gold": "Paxos",
    "franklin onchain u.s. government money fund": "Franklin Templeton",
}

def normalize_key(s: Any) -> str:
    s = ("" if s is None else str(s)).strip().lower()
    s = re.sub(r"[\s\-_]+", "-", s)
    s = re.sub(r"[^a-z0-9\-]", "", s)
    return s

def normalize_project_key(name: Any) -> str:
    s = ("" if name is None else str(name)).strip().lower()
    s = re.sub(r"\s+", " ", s)
    return s


# ----------------------------
# ASSET IDENTITY REGISTRY
# ----------------------------

def _read_json_file(path: Path) -> Any:
    return json.loads(path.read_text(encoding='utf-8'))

def _write_json_atomic(path: Path, obj: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(tmp, path)

def load_asset_registry(path: Path) -> Dict[str, Any]:
    """Load registry. Returns {'assets': [...]}.
    If missing or malformed, returns an empty registry.
    """
    try:
        if path.exists():
            data = _read_json_file(path)
            if isinstance(data, dict) and isinstance(data.get('assets'), list):
                return data
            # tolerate legacy 'list root'
            if isinstance(data, list):
                return {'assets': data}
    except Exception:
        pass
    return {'assets': []}

def build_registry_index(registry: Dict[str, Any]) -> Dict[str, str]:
    """Map any known slug/alias -> canonical_id."""
    idx: Dict[str, str] = {}
    for entry in registry.get('assets', []):
        if not isinstance(entry, dict):
            continue
        canonical = str(entry.get('canonical_id') or entry.get('source_slug') or '').strip()
        if not canonical:
            continue
        keys = set()
        if entry.get('source_slug'):
            keys.add(str(entry['source_slug']).strip())
        keys.add(canonical)
        aliases = entry.get('aliases') or []
        if isinstance(aliases, list):
            for a in aliases:
                if a:
                    keys.add(str(a).strip())
        for k in keys:
            idx[k] = canonical
    return idx

def ensure_registry_entry(
    registry: Dict[str, Any],
    canonical_id: str,
    source_slug: str,
    display_name: str,
    category: Optional[str],
    issuer: Optional[str],
    chains: List[str],
    snapshot_date: str,
) -> None:
    assets = registry.setdefault('assets', [])
    # find existing
    found = None
    for e in assets:
        if isinstance(e, dict) and str(e.get('canonical_id') or '') == canonical_id:
            found = e
            break
    if found is None:
        found = {
            'canonical_id': canonical_id,
            'source_slug': source_slug,
            'display_name': display_name,
            'aliases': [],
            'category': category,
            'issuer': issuer,
            'chains': chains,
            'first_seen': snapshot_date,
            'last_seen': snapshot_date,
            'status': 'active',
        }
        assets.append(found)
    else:
        # update minimal fields; do not overwrite curated ones if present
        found['last_seen'] = snapshot_date
        if not found.get('display_name'):
            found['display_name'] = display_name
        if not found.get('source_slug'):
            found['source_slug'] = source_slug
        if found.get('category') in (None, '', 'Other RWA') and category:
            found['category'] = category
        if not found.get('issuer') and issuer:
            found['issuer'] = issuer
        if not found.get('chains') and chains:
            found['chains'] = chains
        if found.get('status') is None:
            found['status'] = 'active'

def resolve_asset_identity(
    source_slug: str,
    display_name: str,
    category: Optional[str],
    issuer: Optional[str],
    chains: List[str],
    snapshot_date: str,
    registry: Dict[str, Any],
    registry_idx: Dict[str, str],
) -> Tuple[str, str]:
    """Return (asset_id, asset_key). Uses registry canonical_id if available, else falls back to slug.
    asset_key == asset_id for now.
    """
    canonical = registry_idx.get(source_slug) or source_slug
    canonical = str(canonical).strip() or source_slug
    if AUTO_UPDATE_ASSET_REGISTRY:
        ensure_registry_entry(
            registry=registry,
            canonical_id=canonical,
            source_slug=source_slug,
            display_name=display_name,
            category=category,
            issuer=issuer,
            chains=chains,
            snapshot_date=snapshot_date,
        )
    return canonical, canonical

def fetch_protocols() -> List[Dict[str, Any]]:
    data = get_json(DEFI_LLAMA_PROTOCOLS_URL)
    if not isinstance(data, list):
        raise RuntimeError("Unexpected response from /protocols")
    return data

def fetch_yield_pools() -> List[Dict[str, Any]]:
    data = get_json(DEFI_LLAMA_YIELDS_POOLS_URL)
    pools = data.get("data", []) if isinstance(data, dict) else []
    if not isinstance(pools, list):
        return []
    return pools

def aggregate_yield_integrity_by_project(pools: List[Dict[str, Any]]) -> Dict[str, YieldIntegrity]:
    grouped: Dict[str, Dict[str, Any]] = {}
    for pool in pools:
        project = pool.get("project")
        key = normalize_project_key(project)
        if not key:
            continue

        tvl = safe_float(pool.get("tvlUsd")) or 0.0
        apy = safe_float(pool.get("apy"))

        if tvl < POOL_TVL_CUTOFF_USD:
            continue
        if apy is not None and apy > APY_EXCLUDE_GT:
            continue

        g = grouped.setdefault(key, {
            "sum_pool_tvl": 0.0,
            "pool_count": 0,
            "reported_high": False,
            "reported_strong": False,
        })
        g["sum_pool_tvl"] += tvl
        g["pool_count"] += 1
        if apy is not None and apy > APY_WARNING_GT:
            g["reported_high"] = True
        if apy is not None and apy > APY_STRONG_WARNING_GT:
            g["reported_strong"] = True

    out: Dict[str, YieldIntegrity] = {}
    for key, g in grouped.items():
        pool_count = int(g["pool_count"])
        if pool_count >= 4:
            confidence = "HIGH"
        elif pool_count >= 2:
            confidence = "MEDIUM"
        else:
            confidence = "LOW"
        out[key] = YieldIntegrity(
            confidence=confidence,
            sum_pool_tvl=float(g["sum_pool_tvl"]),
            pool_count_used=pool_count,
            reported_high_apy_detected=bool(g["reported_high"]),
            reported_strong_apy_detected=bool(g["reported_strong"]),
        )
    return out

def build_yield_alias_index(pools: List[Dict[str, Any]]) -> Dict[str, str]:
    alias_to_key: Dict[str, str] = {}
    for pool in pools:
        project = pool.get("project")
        key = normalize_project_key(project)
        if not key:
            continue
        candidates = {
            normalize_project_key(project),
            normalize_project_key(pool.get("symbol")),
            normalize_project_key(pool.get("poolMeta")),
            normalize_project_key(pool.get("chain")),
        }
        for c in list(candidates):
            if c:
                alias_to_key[c] = key
                alias_to_key[normalize_key(c)] = key
    return alias_to_key

def match_yield_integrity_to_protocol(protocol: Dict[str, Any], integrity_by_project_key: Dict[str, YieldIntegrity], alias_to_project_key: Dict[str, str]) -> Optional[YieldIntegrity]:
    candidates = [
        normalize_project_key(protocol.get("name")),
        normalize_project_key(protocol.get("slug")),
        normalize_project_key(protocol.get("symbol")),
        normalize_project_key(protocol.get("url")),
    ]
    for c in list(candidates):
        if not c:
            continue
        if c in integrity_by_project_key:
            return integrity_by_project_key[c]
        if c in alias_to_project_key and alias_to_project_key[c] in integrity_by_project_key:
            return integrity_by_project_key[alias_to_project_key[c]]
        nk = normalize_key(c)
        if nk in alias_to_project_key and alias_to_project_key[nk] in integrity_by_project_key:
            return integrity_by_project_key[alias_to_project_key[nk]]
    return None

def map_protocol_category(raw_category: Any) -> str:
    raw = ("" if raw_category is None else str(raw_category)).strip()
    if not raw:
        return "Other"
    return PROTOCOL_CATEGORY_MAP.get(raw.lower(), raw)

def infer_issuer(name: Any) -> Optional[str]:
    k = normalize_project_key(name)
    return KNOWN_RWA_ASSET_NAME_TO_ISSUER.get(k)


def infer_asset_type(analytics_category: Any) -> str:
    """Coarse asset type taxonomy derived from analytics_category (stable across time)."""
    c = ("" if analytics_category is None else str(analytics_category)).strip().lower()
    if c == "treasury":
        return "tokenized_treasury"
    if c == "private credit":
        return "private_credit"
    if c == "real estate":
        return "real_estate"
    if c == "commodities":
        return "commodity_backed"
    if c == "stablecoins":
        return "stablecoin_rwa"
    if c == "funds":
        return "fund"
    return "other_rwa"

def pick_primary_chain(chains: Any) -> Optional[str]:
    if not chains:
        return None
    if isinstance(chains, str):
        return chains
    try:
        lst = [str(x) for x in chains if x]
        if not lst:
            return None
        # Stable choice: alphabetical
        return sorted(set(lst))[0]
    except Exception:
        return None

def infer_issuer_id(issuer_name: Any) -> Optional[str]:
    if not issuer_name:
        return None
    return normalize_key(issuer_name)

def build_rwa_assets_df(protocols: List[Dict[str, Any]], pools: List[Dict[str, Any]]) -> pd.DataFrame:
    rwa_protocols = [p for p in protocols if str(p.get("category") or "").strip() == "RWA"]

    integrity_by_project_key = aggregate_yield_integrity_by_project(pools)
    alias_to_project_key = build_yield_alias_index(pools)

    rows: List[Dict[str, Any]] = []
    for p in rwa_protocols:
        name = p.get("name") or "N/A"
        slug = normalize_key(p.get("slug") or slugify(name))
        tvl = safe_float(p.get("tvl")) or 0.0
        change_7d = safe_float(p.get("change_7d"))
        chains = p.get("chains") or []
        if not isinstance(chains, list):
            chains = []

        integ = match_yield_integrity_to_protocol(p, integrity_by_project_key, alias_to_project_key)
        if integ is not None:
            coverage_pct = None
            if tvl > 0 and integ.sum_pool_tvl > 0:
                coverage_pct = max(0.0, min(100.0, (integ.sum_pool_tvl / tvl) * 100.0))
            confidence = integ.confidence
            reported_high = integ.reported_high_apy_detected
        else:
            coverage_pct = None
            confidence = "LOW"
            reported_high = False

        row = {
            "slug": slug,
            "id": p.get("id"),
            "name": name,
            "tvl_usd": tvl,
            "protocol_category": map_protocol_category(p.get("category")),
            "flag_low_yield_coverage": bool(coverage_pct is not None and coverage_pct < LOW_COVERAGE_THRESHOLD_PCT),
            "reported_high_apy_detected": bool(reported_high),
            "category": p.get("category") or "RWA",
            "asset_class": None,
            "access_model": None,
            "type": None,
            "rwa_classification": None,
            "issuer": infer_issuer(name),
            "issuer_id": infer_issuer_id(infer_issuer(name)),
            "chains": chains,
            "active_marketcap_usd": None,
            "onchain_marketcap_usd": None,
            "token_price": None,
            "utilization": None,
            "redeemable": None,
            "attestations": None,
            "cex_listed": None,
            "kyc_mint_redeem": None,
            "kyc_transfer_hold": None,
            "transferable": None,
            "self_custody": None,
            "monitored_value_usd": tvl,
            "tvl_change_7d_pct": change_7d,
            "confidence": confidence,
            "yield_coverage_pct": coverage_pct,
            "defillama_url": f"https://defillama.com/protocol/{slug}",
            "project_url": p.get("url") or None,
            "logo": p.get("logo"),
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    if df.empty:
        raise RuntimeError("No RWA rows were produced from the /protocols dataset.")

    df["tvl_usd"] = df["tvl_usd"].apply(safe_float).fillna(0.0)
    df = df.sort_values(["tvl_usd", "slug"], ascending=[False, True]).reset_index(drop=True)
    return df

protocols_raw = fetch_protocols()
yield_pools_raw = fetch_yield_pools()

# Fail-fast sanity checks: if upstream API shape changes or returns empty payloads,
# do NOT produce a snapshot (prevents silently writing bad history).
if not isinstance(protocols_raw, list) or len(protocols_raw) == 0:
    raise RuntimeError("DefiLlama protocols payload is empty or not a list (upstream API change / outage).")
if yield_pools_raw is None:
    # yield_pools is optional for core TVL, but if fetch failed hard we should flag it.
    print("[WARN] Yield pools payload is None; monitoring flags may be degraded.")
elif not isinstance(yield_pools_raw, dict):
    print("[WARN] Yield pools payload is not a dict; monitoring flags may be degraded.")
assets_df = build_rwa_assets_df(protocols_raw, yield_pools_raw)

# Compatibility variables kept for downstream cells.
href_map = {}
csv_links = []
rwa_html = ""
raw_rwa_df = assets_df.copy()

print("Protocols fetched:", len(protocols_raw))
print("Yield pools fetched:", len(yield_pools_raw))
print("RWA rows built:", len(assets_df))
display(assets_df.head(10))



# Investor-facing analytics taxonomy
ANALYTICS_CATEGORY_OVERRIDES = {
    "tether-gold": "Commodities",
    "paxos-gold": "Commodities",
    "gold-dao": "Commodities",
    "ondo-global-markets": "Treasury",
    "blackrock-buidl": "Treasury",
    "superstate-ustb": "Treasury",
    "ethena-usdtb": "Stablecoins",
    "circle-usyc": "Stablecoins",
    "centrifuge-protocol": "Private Credit",
    "realtyx": "Real Estate",
    "chateau": "Real Estate",
    "propbase": "Real Estate",
    "spiko": "Treasury",
    "wisdomtree": "Funds",
}

ANALYTICS_CATEGORY_RULES = [
    ("Treasury", ["treasury", "tbill", "t-bill", "ustb", "buidl", "government money", "money fund", "treasuries", "spiko", "benji", "ousg"]),
    ("Stablecoins", ["stablecoin", "stablecoins", "usdtb", "usyc", "usd0", "usdr", "eusd", "fusd"]),
    ("Private Credit", ["private credit", "credit", "trade finance", "invoice", "centrifuge", "fortunafi", "credix", "maple"]),
    ("Real Estate", ["real estate", "realty", "estate", "property", "propbase", "chateau", "realt"]),
    ("Commodities", ["gold", "silver", "commodity", "commodities", "xau", "paxg"]),
    ("Funds", ["fund", "yield", "income", "note", "capital", "markets"]),
]

def infer_analytics_category(name: Any, slug: Any, issuer: Any = None) -> str:
    slug_k = normalize_key(slug)
    if slug_k in ANALYTICS_CATEGORY_OVERRIDES:
        return ANALYTICS_CATEGORY_OVERRIDES[slug_k]

    hay = " ".join([
        str(name or ""),
        str(slug or ""),
        str(issuer or ""),
    ]).strip().lower()

    for cat, keywords in ANALYTICS_CATEGORY_RULES:
        if any(k in hay for k in keywords):
            return cat
    return "Other RWA"

assets_df["analytics_category"] = assets_df.apply(lambda r: infer_analytics_category(r.get("name"), r.get("slug"), r.get("issuer")), axis=1)

# Stable identifiers / taxonomy enrichment (do NOT affect TVL measurement).
# asset_id: prefer DefiLlama numeric id if present; otherwise fall back to slug.

# ---- Identity / taxonomy enrichment ----
# issuer_id / asset_type / primary_chain are derived fields and can be re-computed,
# but we also store them into the snapshot/CSV for convenience.
if "issuer_id" not in assets_df.columns:
    assets_df["issuer_id"] = assets_df["issuer"].apply(infer_issuer_id)

if "asset_type" not in assets_df.columns:
    assets_df["asset_type"] = assets_df["analytics_category"].apply(infer_asset_type)

if "primary_chain" not in assets_df.columns:
    assets_df["primary_chain"] = assets_df["chains"].apply(pick_primary_chain)

# Stable asset identity: prefer canonical_id from the registry when available.
asset_registry = load_asset_registry(ASSET_REGISTRY_PATH)
asset_registry_idx = build_registry_index(asset_registry)

# Track registry state before resolving identities (for audit/validation reporting).
_pre_registry_ids = set()
try:
    _pre_registry_ids = {str(e.get("canonical_id") or "").strip() for e in (asset_registry or []) if str(e.get("canonical_id") or "").strip()}
except Exception:
    _pre_registry_ids = set()


def _resolve_row_identity(r: pd.Series) -> Tuple[str, str]:
    slug = str(r.get("slug") or "").strip()
    name = str(r.get("name") or "").strip()
    cat = str(r.get("analytics_category") or r.get("protocol_category") or "").strip() or None
    issuer = r.get("issuer")
    chains = r.get("chains") or []
    if not isinstance(chains, list):
        chains = []
    return resolve_asset_identity(
        source_slug=slug,
        display_name=name,
        category=cat,
        issuer=issuer,
        chains=chains,
        snapshot_date=SNAPSHOT_DATE_STR,
        registry=asset_registry,
        registry_idx=asset_registry_idx,
    )

assets_df[["asset_id", "asset_key"]] = assets_df.apply(lambda r: pd.Series(_resolve_row_identity(r)), axis=1)

# Build identity registry report for this run (new assets, alias matches, conflicts).
IDENTITY_REPORT = {
    "date": SNAPSHOT_DATE_STR,
    "new_assets": [],
    "alias_matches": [],
    "conflicts": [],
}
try:
    # New canonical IDs introduced this run (relative to the registry before updates).
    current_ids = set(str(x).strip() for x in assets_df.get("asset_id", pd.Series([])).dropna().tolist() if str(x).strip())
    IDENTITY_REPORT["new_assets"] = sorted(current_ids - _pre_registry_ids)

    # Alias matches: source slug differs from canonical_id (indicates registry alias mapping).
    if "slug" in assets_df.columns and "asset_id" in assets_df.columns:
        alias_df = assets_df.loc[assets_df["slug"].astype(str) != assets_df["asset_id"].astype(str), ["slug", "asset_id", "name"]].copy()
        IDENTITY_REPORT["alias_matches"] = [
            {"source_slug": str(r["slug"]), "canonical_id": str(r["asset_id"]), "name": str(r.get("name") or "")}
            for _, r in alias_df.iterrows()
        ]

    # Conflicts: multiple distinct source slugs mapping to the same canonical_id in the same snapshot.
    if "slug" in assets_df.columns and "asset_id" in assets_df.columns:
        grp = assets_df.groupby("asset_id")["slug"].nunique()
        conflict_ids = grp[grp > 1].index.tolist()
        for cid in conflict_ids:
            slugs = sorted(set(assets_df.loc[assets_df["asset_id"] == cid, "slug"].astype(str).tolist()))
            IDENTITY_REPORT["conflicts"].append({"canonical_id": str(cid), "source_slugs": slugs})
except Exception as e:
    print("[WARN] Failed to build IDENTITY_REPORT:", e)

# Persist registry updates (first_seen/last_seen/new assets) if enabled and the path exists or can be created.
if AUTO_UPDATE_ASSET_REGISTRY:
    try:
        _write_json_atomic(ASSET_REGISTRY_PATH, asset_registry)
    except Exception as e:
        print("[WARN] Failed to write asset registry:", e)

display(assets_df[["name", "slug", "protocol_category", "analytics_category", "tvl_usd"]].head(15))


In [ ]:
# ----------------------------
# SNAPSHOT BUILD
# ----------------------------

def build_snapshot_from_assets(df: pd.DataFrame, snapshot_date: str) -> Dict[str, Any]:
    out_assets = []
    for _, row in df.iterrows():
        out_assets.append({
            "slug": row["slug"],
            "asset_id": row.get("asset_id") or row["slug"],
            "asset_key": row.get("asset_key") or row.get("asset_id") or row["slug"],
            "name": row["name"],
            "tvl_usd": safe_float(row["tvl_usd"]) or 0.0,
            "protocol_category": row["protocol_category"],
            "analytics_category": row.get("analytics_category") or "Other RWA",
            "flag_low_yield_coverage": bool(row["flag_low_yield_coverage"]),
            "reported_high_apy_detected": bool(row["reported_high_apy_detected"]),
            "issuer": row.get("issuer"),
            "issuer_id": row.get("issuer_id"),
            "asset_type": row.get("asset_type"),
            "primary_chain": row.get("primary_chain"),
            "chains": row.get("chains") or [],
            "category": row.get("category"),
            "asset_class": row.get("asset_class"),
            "access_model": row.get("access_model"),
            "type": row.get("type"),
            "rwa_classification": row.get("rwa_classification"),
            "active_marketcap_usd": safe_float(row.get("active_marketcap_usd")),
            "onchain_marketcap_usd": safe_float(row.get("onchain_marketcap_usd")),
            "token_price": safe_float(row.get("token_price")),
            "utilization": safe_float(row.get("utilization")),
            "redeemable": row.get("redeemable"),
            "attestations": row.get("attestations"),
            "cex_listed": row.get("cex_listed"),
            "kyc_mint_redeem": row.get("kyc_mint_redeem"),
            "kyc_transfer_hold": row.get("kyc_transfer_hold"),
            "transferable": row.get("transferable"),
            "self_custody": row.get("self_custody"),
        })

    snapshot = {
        "generated_at": iso_ts(datetime.now(UTC)),
        "version": "rwa-watch-v6-analytics-1.0",
        "date": snapshot_date,
        "assets": out_assets,
        "source": {
            "provider": "DefiLlama",
            "public_page": RWA_PUBLIC_PAGE_URL,
            "api_endpoints": [DEFI_LLAMA_PROTOCOLS_URL, DEFI_LLAMA_YIELDS_POOLS_URL],
            "csv_links_discovered": csv_links,
        }
    }
    return snapshot

snapshot = build_snapshot_from_assets(assets_df, SNAPSHOT_DATE_STR)

# Use an immutable filename: if today's snapshot already exists, append __HHMMSS (UTC).
base_snapshot_path = SNAPSHOT_ROOT / f"{SNAPSHOT_DATE_STR}.json"
snapshot_path = unique_path_if_exists(base_snapshot_path)

# Archive raw source payloads for auditability (stored under _private/sources/<snapshot_id>/).
snapshot_id = snapshot_path.stem
raw_archives = save_source_archives(snapshot_id, protocols_raw, yield_pools_raw)

# Write snapshot atomically + embed checksums/paths into metadata.
snapshot["source"]["raw_archives"] = raw_archives
snapshot_sha256 = write_json(snapshot_path, snapshot)
snapshot["source"]["snapshot_sha256"] = snapshot_sha256
# Re-write once more to persist the embedded sha (still atomic).
snapshot_sha256 = write_json(snapshot_path, snapshot)
snapshot["source"]["snapshot_sha256"] = snapshot_sha256

# Maintain a stable pointer for the UI or quick inspection.
latest_path = SNAPSHOT_ROOT / "latest.json"
write_json(latest_path, snapshot)

print("Saved analytics snapshot to:", snapshot_path)
print("Raw sources archived under:", (PRIVATE_DATA_ROOT / "sources" / snapshot_id))


In [ ]:
# ----------------------------
# VALIDATION
# ----------------------------

def validate_snapshot(snapshot: Dict[str, Any]) -> Dict[str, Any]:
    errors = []
    warnings = []
    assets = snapshot.get("assets", [])

    if not isinstance(snapshot.get("generated_at"), str):
        errors.append("generated_at missing or not a string")
    if not isinstance(snapshot.get("date"), str):
        errors.append("date missing or not a string")
    if not isinstance(assets, list):
        errors.append("assets missing or not a list")
        assets = []

    slugs = [a.get("slug") for a in assets]
    names = [a.get("name") for a in assets]

    duplicate_slugs = sorted({s for s in slugs if s and slugs.count(s) > 1})
    missing_name = [a.get("slug") for a in assets if not str(a.get("name") or "").strip()]
    negative_tvl = [a.get("slug") for a in assets if safe_float(a.get("tvl_usd")) is not None and safe_float(a.get("tvl_usd")) < 0]

    if duplicate_slugs:
        errors.append(f"duplicate slug count: {len(duplicate_slugs)}")
    if missing_name:
        errors.append(f"missing name count: {len(missing_name)}")
    if negative_tvl:
        warnings.append(f"negative tvl count: {len(negative_tvl)}")

    total_tvl = 0.0
    asset_tvl_count = 0
    for a in assets:
        v = safe_float(a.get("tvl_usd"))
        if v is not None:
            total_tvl += v
            asset_tvl_count += 1

    if asset_tvl_count == 0:
        errors.append("no numeric tvl values found")

    validation = {
        "date": snapshot.get("date"),
        "generated_at": iso_ts(datetime.now(UTC)),
        "status": "ok" if not errors else "error",
        "checks": {
            "schema_validation": "ok" if not errors else "error",
            "duplicate_slug": {"status": "ok" if not duplicate_slugs else "error", "items": duplicate_slugs},
            "missing_name": {"status": "ok" if not missing_name else "error", "items": missing_name},
            "negative_tvl": {"status": "ok" if not negative_tvl else "warning", "items": negative_tvl},
            "reconciliation": {"status": "ok", "total_tvl": total_tvl, "asset_tvl_count": asset_tvl_count},
            "anomaly_detection": {"status": "ok", "notes": []},
        },
        "errors": errors,
        "warnings": warnings,
    }
    return validation

validation = validate_snapshot(snapshot)

# Attach identity registry report (new assets, alias matches, conflicts) when available.
try:
    if "IDENTITY_REPORT" in globals() and isinstance(IDENTITY_REPORT, dict):
        validation["checks"]["identity_registry"] = {
            "status": "ok" if not (IDENTITY_REPORT.get("conflicts") or []) else "warning",
            "new_assets_count": len(IDENTITY_REPORT.get("new_assets") or []),
            "alias_matches_count": len(IDENTITY_REPORT.get("alias_matches") or []),
            "conflicts_count": len(IDENTITY_REPORT.get("conflicts") or []),
            "details": IDENTITY_REPORT,
        }
except Exception as e:
    validation["checks"]["identity_registry"] = {"status": "warning", "notes": [f"failed to attach IDENTITY_REPORT: {e}"]}

# Anomaly detection vs previous snapshot (fail-fast on extreme discontinuities).
def _load_snapshot_for_date(date_str: str) -> Optional[Dict[str, Any]]:
    # Choose the latest immutable file for that logical date (YYYY-MM-DD or YYYY-MM-DD__HHMMSS).
    candidates = sorted(SNAPSHOT_ROOT.glob(f"{date_str}*.json"))
    if not candidates:
        return None
    # Prefer the lexicographically last (HHMMSS suffix sorts correctly).
    path = candidates[-1]
    try:
        return read_json(path)
    except Exception as exc:
        print("[WARN] Unable to read previous snapshot:", path, exc)
        return None

def _snapshot_total_tvl(payload: Dict[str, Any]) -> float:
    total = 0.0
    for a in payload.get("assets", []) or []:
        v = safe_float(a.get("tvl_usd"))
        if v is not None:
            total += v
    return float(total)

try:
    today_total = float(validation["checks"]["reconciliation"]["total_tvl"])
    prev_date = (datetime.fromisoformat(SNAPSHOT_DATE_STR).date() - timedelta(days=1)).isoformat()
    prev_payload = _load_snapshot_for_date(prev_date)
    notes = []
    status = "ok"

    if prev_payload:
        prev_total = _snapshot_total_tvl(prev_payload)
        pct = (abs(today_total - prev_total) / prev_total) if prev_total > 0 else (0.0 if today_total == 0 else 1.0)
        notes.append({
            "prev_date": prev_date,
            "prev_total_tvl": prev_total,
            "today_total_tvl": today_total,
            "abs_change": today_total - prev_total,
            "pct_change": pct,
        })

        # Soft warning threshold: 50% move day-over-day is rare; record as warning.
        if pct >= 0.50:
            status = "warning"
            validation.setdefault("warnings", []).append(f"anomaly: day-over-day total TVL changed by {pct:.1%} vs {prev_date}")

        # Hard fail threshold: >=90% likely indicates outage/schema change; stop the pipeline.
        if pct >= 0.90:
            status = "error"
            validation.setdefault("errors", []).append(f"hard anomaly: day-over-day total TVL changed by {pct:.1%} vs {prev_date} (likely upstream issue).")
    else:
        notes.append({"prev_date": prev_date, "note": "previous-day snapshot not found; skipping day-over-day anomaly check"})

    validation["checks"]["anomaly_detection"] = {"status": status, "notes": notes}
    # Promote overall status if anomaly is hard error.
    if status == "error":
        validation["status"] = "error"
except Exception as exc:
    validation["checks"]["anomaly_detection"] = {"status": "warning", "notes": [f"failed anomaly detection: {exc}"]}

# Fail-fast if validation has hard errors (prevents persisting derived outputs from bad snapshots).
if validation.get("status") == "error" or (validation.get("errors") or []):
    raise RuntimeError("Snapshot validation failed: " + "; ".join(validation.get("errors") or []))
validation_path = VALIDATION_ROOT / f"{SNAPSHOT_DATE_STR}.json"
write_json(validation_path, validation)

# Public mirror for auditability inside the analysis site
site_validation_path = SITE_VALIDATION_ROOT / f"{SNAPSHOT_DATE_STR}.json"
write_json(site_validation_path, validation)

print("Validation saved:", validation_path)
print(json.dumps(validation, indent=2)[:900], "...")


In [ ]:
# ----------------------------
# HISTORY BUILD FROM PRIOR SNAPSHOTS
# ----------------------------

def load_all_snapshots(snapshot_dir: Path) -> Dict[str, Dict[str, Any]]:
    out = {}
    for path in sorted(snapshot_dir.glob("*.json")):
        try:
            payload = read_json(path)
            d = payload.get("date")
            if d:
                out[d] = payload
        except Exception as exc:
            print("Skipping unreadable snapshot:", path, exc)
    return dict(sorted(out.items()))

all_snapshots = load_all_snapshots(SNAPSHOT_ROOT)

def snapshot_to_df(snapshot_payload: Dict[str, Any]) -> pd.DataFrame:
    df = pd.DataFrame(snapshot_payload.get("assets", []))
    if df.empty:
        return pd.DataFrame(columns=["slug", "name", "tvl_usd", "protocol_category", "analytics_category"])
    df["date"] = snapshot_payload["date"]
    return df

history_frames = [snapshot_to_df(s) for s in all_snapshots.values()]
history_df = pd.concat(history_frames, ignore_index=True) if history_frames else pd.DataFrame()

if not history_df.empty:
    history_df["tvl_usd"] = history_df["tvl_usd"].apply(safe_float).fillna(0.0)
    history_df["rank_by_tvl"] = history_df.groupby("date")["tvl_usd"].rank(method="first", ascending=False).astype(int)

display(history_df.head(10))
print("History dates available:", list(all_snapshots.keys())[-10:])


if not history_df.empty and "analytics_category" not in history_df.columns:
    history_df["analytics_category"] = "Other RWA"


In [ ]:

# ----------------------------
# DAILY / WEEKLY METRICS
# ----------------------------

ELIGIBLE_THRESHOLD_USD = 1_000_000.0

def get_day_df(day: str) -> pd.DataFrame:
    frame = history_df.loc[history_df["date"] == day].copy()
    if frame.empty:
        return pd.DataFrame(columns=history_df.columns)
    frame["tvl_usd"] = frame["tvl_usd"].fillna(0.0)
    if "analytics_category" not in frame.columns:
        frame["analytics_category"] = "Other RWA"
    # Deterministic ranking: sort first, then assign ranks 1..N
    frame = frame.sort_values(["tvl_usd", "slug"], ascending=[False, True]).reset_index(drop=True)
    frame["rank_by_tvl"] = np.arange(1, len(frame) + 1)
    total = float(frame["tvl_usd"].sum())
    frame["share"] = (frame["tvl_usd"] / total) if total else 0.0
    return frame

today_df = get_day_df(SNAPSHOT_DATE_STR)
yesterday_date = (SNAPSHOT_DATE - timedelta(days=1)).isoformat()
week_ago_date = (SNAPSHOT_DATE - timedelta(days=7)).isoformat()
yesterday_df = get_day_df(yesterday_date)
week_ago_df = get_day_df(week_ago_date)

# Guardrails: if prior snapshots are missing, do not fabricate deltas
yesterday_available = not yesterday_df.empty
week_ago_available = not week_ago_df.empty


def sum_tvl(df: pd.DataFrame) -> float:
    return float(df["tvl_usd"].sum()) if not df.empty else 0.0

def keyed(df: pd.DataFrame) -> Dict[str, Dict[str, Any]]:
    if df.empty:
        return {}
    return {row["slug"]: row.to_dict() for _, row in df.iterrows()}

today_map = keyed(today_df)
yesterday_map = keyed(yesterday_df)
week_ago_map = keyed(week_ago_df)

today_total = sum_tvl(today_df)
yesterday_total = sum_tvl(yesterday_df)
week_ago_total = sum_tvl(week_ago_df)

today_df["change_1d_usd"] = today_df["slug"].map(
    lambda s: None if not yesterday_available else today_map[s]["tvl_usd"] - safe_float(yesterday_map.get(s, {}).get("tvl_usd") or 0.0)
)
today_df["change_1d_pct"] = today_df["slug"].map(
    lambda s: None if (not yesterday_available) or safe_float(yesterday_map.get(s, {}).get("tvl_usd")) in (None, 0.0) else ((today_map[s]["tvl_usd"] / safe_float(yesterday_map[s]["tvl_usd"])) - 1.0) * 100.0
)
today_df["change_7d_usd"] = today_df["slug"].map(
    lambda s: None if not week_ago_available else today_map[s]["tvl_usd"] - safe_float(week_ago_map.get(s, {}).get("tvl_usd") or 0.0)
)
today_df["change_7d_pct"] = today_df["slug"].map(
    lambda s: None if (not week_ago_available) or safe_float(week_ago_map.get(s, {}).get("tvl_usd")) in (None, 0.0) else ((today_map[s]["tvl_usd"] / safe_float(week_ago_map[s]["tvl_usd"])) - 1.0) * 100.0
)

today_df["eligible_today"] = today_df["tvl_usd"] >= ELIGIBLE_THRESHOLD_USD
today_df["in_top500_today"] = (today_df["rank_by_tvl"] <= 500) & (today_df["tvl_usd"] >= ELIGIBLE_THRESHOLD_USD)

eligible_today_df = today_df.loc[today_df["eligible_today"]].copy()
# Market breadth is computed for internal monitoring only; treat as unreliable until enough history exists.
MIN_ELIGIBLE_FOR_BREADTH = 10
if (not week_ago_available) or eligible_today_df.empty or len(eligible_today_df) < MIN_ELIGIBLE_FOR_BREADTH:
    breadth = None
else:
    # 7D breadth is less noisy than 24H breadth
    positive_assets = int((eligible_today_df["change_7d_usd"] > 0).sum())
    breadth = (positive_assets / int(len(eligible_today_df))) if len(eligible_today_df) else None

top3 = today_df.head(3).copy()
top10 = today_df.head(10).copy()
top3_concentration = float(top3["tvl_usd"].sum() / today_total) if today_total else None
top10_concentration = float(top10["tvl_usd"].sum() / today_total) if today_total else None
top3_assets = top3["name"].tolist()

# Top 500 aggregate TVL (ranked by TVL, deterministic ordering). This is useful for long-term market tracking.
top500_total_today = float(today_df.head(500)["tvl_usd"].sum()) if not today_df.empty else 0.0
top500_total_yesterday = float(yesterday_df.head(500)["tvl_usd"].sum()) if yesterday_available and (not yesterday_df.empty) else None
top500_total_week_ago = float(week_ago_df.head(500)["tvl_usd"].sum()) if week_ago_available and (not week_ago_df.empty) else None

top500_change_1d_usd = (top500_total_today - top500_total_yesterday) if top500_total_yesterday is not None else None
top500_change_1d_pct = (((top500_total_today / top500_total_yesterday) - 1.0) * 100.0) if (top500_total_yesterday is not None and top500_total_yesterday) else None
top500_change_7d_usd = (top500_total_today - top500_total_week_ago) if top500_total_week_ago is not None else None
top500_change_7d_pct = (((top500_total_today / top500_total_week_ago) - 1.0) * 100.0) if (top500_total_week_ago is not None and top500_total_week_ago) else None

def category_change_map(base_df: pd.DataFrame, compare_df: pd.DataFrame, category_col: str = "analytics_category") -> pd.DataFrame:
    compare_available = not compare_df.empty
    cats = sorted(set(base_df.get(category_col, pd.Series(dtype=object)).dropna().astype(str).tolist()) |
                  set(compare_df.get(category_col, pd.Series(dtype=object)).dropna().astype(str).tolist()))
    rows = []
    total_now = float(base_df["tvl_usd"].sum()) if not base_df.empty else 0.0
    for cat in cats:
        today_v = float(base_df.loc[base_df[category_col] == cat, "tvl_usd"].sum()) if not base_df.empty else 0.0
        prev_v = float(compare_df.loc[compare_df[category_col] == cat, "tvl_usd"].sum()) if compare_available else None
        rows.append({
            "category": cat,
            "tvl_usd": today_v,
            "share": (today_v / total_now) if total_now else None,
            "change_usd": None if prev_v is None else (today_v - prev_v),
            "change_pct": None if prev_v in (None, 0.0) else ((today_v / prev_v) - 1.0) * 100.0,
        })
    out = pd.DataFrame(rows)
    if out.empty:
        return pd.DataFrame(columns=["category", "tvl_usd", "share", "change_usd", "change_pct"])
    return out.sort_values(["tvl_usd", "category"], ascending=[False, True]).reset_index(drop=True)

category_1d_df = category_change_map(today_df, yesterday_df, "analytics_category")
category_7d_df = category_change_map(today_df, week_ago_df, "analytics_category")

if yesterday_available:
    largest_capital_increase_row = category_1d_df.dropna(subset=["change_usd"]).sort_values("change_usd", ascending=False).head(1)
    strongest_category_growth_row = category_1d_df.dropna(subset=["change_pct"]).sort_values("change_pct", ascending=False).head(1)
    largest_asset_increase_row = eligible_today_df.dropna(subset=["change_1d_usd"]).sort_values("change_1d_usd", ascending=False).head(1)
    largest_asset_decline_row = eligible_today_df.dropna(subset=["change_1d_usd"]).sort_values("change_1d_usd", ascending=True).head(1)
else:
    largest_capital_increase_row = pd.DataFrame()
    strongest_category_growth_row = pd.DataFrame()
    largest_asset_increase_row = pd.DataFrame()
    largest_asset_decline_row = pd.DataFrame()


if yesterday_available:
    new_entrants_df = today_df.loc[
        (today_df["tvl_usd"] >= ELIGIBLE_THRESHOLD_USD)
        & (today_df["slug"].map(lambda s: safe_float(yesterday_map.get(s, {}).get("tvl_usd")) or 0.0) < ELIGIBLE_THRESHOLD_USD)
    ].copy()

    new_top10_entrants_df = today_df.loc[
        (today_df["rank_by_tvl"] <= 10)
        & (today_df["slug"].map(lambda s: int(yesterday_map.get(s, {}).get("rank_by_tvl", 999999))) > 10)
    ].copy()
else:
    new_entrants_df = today_df.iloc[0:0].copy()
    new_top10_entrants_df = today_df.iloc[0:0].copy()

# Broader window so the "Top 10 Rank Watch" can actually fill out
near_top10_df = today_df.loc[(today_df["rank_by_tvl"] >= 6) & (today_df["rank_by_tvl"] <= 14)].copy()
near_top10_df["previous_rank"] = near_top10_df["slug"].map(lambda s: int(yesterday_map.get(s, {}).get("rank_by_tvl", 999999))) if yesterday_available else None
near_top10_df["rank_change"] = (near_top10_df["previous_rank"] - near_top10_df["rank_by_tvl"]) if yesterday_available else None
near_top10_df = near_top10_df.sort_values(["rank_by_tvl", "tvl_usd"], ascending=[True, False]).reset_index(drop=True)

def build_rank_watch(today_df: pd.DataFrame) -> List[Dict[str, Any]]:
    """Top 10 leaderboard + rank movement vs yesterday (if available)."""
    rows: List[Dict[str, Any]] = []
    top10_df = today_df.head(10).copy()
    for _, row in top10_df.iterrows():
        if yesterday_available:
            prev_rank = int(yesterday_map.get(row["slug"], {}).get("rank_by_tvl", 999999))
        else:
            prev_rank = None

        if prev_rank is None:
            status = "No prior snapshot"
        elif prev_rank > 10:
            status = f"Entered Top 10 (from #{int(prev_rank)})"
        elif int(row["rank_by_tvl"]) < prev_rank:
            status = f"Moved up from #{int(prev_rank)}"
        elif int(row["rank_by_tvl"]) > prev_rank:
            status = f"Moved down from #{int(prev_rank)}"
        else:
            status = f"Held rank #{int(prev_rank)}"

        rows.append({
            "name": row["name"],
            "slug": row["slug"],
            "rank_by_tvl": int(row["rank_by_tvl"]),
            "previous_rank": prev_rank if prev_rank is None else int(prev_rank),
            "tvl_usd": float(row["tvl_usd"]),
            "status": status,
        })
    return rows

rank_watch_items = build_rank_watch(today_df) if yesterday_available else []

def build_market_regime(metrics: Dict[str, Any]) -> str:
    # Public-facing regime should rely only on robust signals.
    top3c = metrics.get("top3_concentration")
    ch7d = metrics.get("change_7d_pct")
    conc_txt = "high concentration" if top3c is not None and top3c >= 45 else "moderate concentration"
    trend_txt = "expansion" if ch7d is not None and ch7d > 0 else "contraction" if ch7d is not None and ch7d < 0 else "mixed conditions"
    return f"{trend_txt.capitalize()} with {conc_txt}."


# ----------------------------
# INVESTOR-GRADE DERIVATIONS (stable, reproducible from gold dataset)
# ----------------------------
protocols_tracked = int(len(today_df))

# Top 3 details (names + share pct)
top3_details = []
for _, r in today_df.head(3).iterrows():
    top3_details.append({
        "slug": r.get("slug"),
        "name": r.get("name"),
        "tvl_usd": safe_float(r.get("tvl_usd")) or 0.0,
        "share_pct": (safe_float(r.get("share")) or 0.0) * 100.0,
        "rank_by_tvl": int(r.get("rank_by_tvl")) if r.get("rank_by_tvl") is not None else None,
    })

# Top 10 breakdown: Top10% and the share of ranks 4-10
top4_10_share_pct = None
if top10_concentration is not None and top3_concentration is not None:
    top4_10_share_pct = (top10_concentration - top3_concentration) * 100.0

# Near Top 10: focus on the #11 asset and the TVL gap to #10
near_top10_focus = None
try:
    if len(today_df) >= 11:
        r10 = today_df.loc[today_df["rank_by_tvl"] == 10].iloc[0] if (today_df["rank_by_tvl"] == 10).any() else None
        r11 = today_df.loc[today_df["rank_by_tvl"] == 11].iloc[0] if (today_df["rank_by_tvl"] == 11).any() else None
        if r10 is not None and r11 is not None:
            tvl10 = safe_float(r10.get("tvl_usd")) or 0.0
            tvl11 = safe_float(r11.get("tvl_usd")) or 0.0
            gap_usd = tvl10 - tvl11
            gap_pct = (gap_usd / tvl10 * 100.0) if tvl10 else None
            prev_rank_11 = int(yesterday_map.get(r11.get("slug"), {}).get("rank_by_tvl", 999999)) if yesterday_available else None
            near_top10_focus = {
                "name": r11.get("name"),
                "slug": r11.get("slug"),
                "rank_by_tvl": 11,
                "tvl_usd": tvl11,
                "gap_to_top10_usd": gap_usd,
                "gap_to_top10_pct": gap_pct,
                "previous_rank": prev_rank_11 if yesterday_available else None,
                "rank_change": (prev_rank_11 - 11) if (yesterday_available and prev_rank_11 is not None and prev_rank_11 < 999999) else None,
            }
except Exception:
    near_top10_focus = None

# Biggest asset gainer (eligible universe) + rank change if available
biggest_asset_gainer = None
if not largest_asset_increase_row.empty:
    rr = largest_asset_increase_row.iloc[0].to_dict()
    prev_rank = int(yesterday_map.get(rr.get("slug"), {}).get("rank_by_tvl", 999999)) if yesterday_available else None
    rank_change = (prev_rank - int(rr.get("rank_by_tvl"))) if (yesterday_available and prev_rank is not None and prev_rank < 999999 and rr.get("rank_by_tvl") is not None) else None
    biggest_asset_gainer = {
        "name": rr.get("name"),
        "slug": rr.get("slug"),
        "tvl_usd": safe_float(rr.get("tvl_usd")) or 0.0,
        "change_1d_usd": safe_float(rr.get("change_1d_usd")),
        "change_1d_pct": safe_float(rr.get("change_1d_pct")),
        "rank_by_tvl": int(rr.get("rank_by_tvl")) if rr.get("rank_by_tvl") is not None else None,
        "previous_rank": prev_rank if yesterday_available else None,
        "rank_change": rank_change,
        "analytics_category": rr.get("analytics_category") or rr.get("category") or "Other RWA",
    }

# Category leader (24h flow) = category with largest absolute +USD change (eligible universe)
category_leader_24h = None
if yesterday_available and not category_1d_df.empty and "change_usd" in category_1d_df.columns:
    best = category_1d_df.dropna(subset=["change_usd"]).sort_values("change_usd", ascending=False).head(1)
    if not best.empty:
        br = best.iloc[0].to_dict()
        category_leader_24h = {
            "category": br.get("category"),
            "change_1d_usd": safe_float(br.get("change_usd")),
            "change_1d_pct": safe_float(br.get("change_pct")),
        }

headline_metrics = {
    "total_rwa_tvl": today_total,
    "top500_rwa_tvl": top500_total_today,
    "top500_change_1d_usd": top500_change_1d_usd,
    "top500_change_1d_pct": top500_change_1d_pct,
    "top500_change_7d_usd": top500_change_7d_usd,
    "top500_change_7d_pct": top500_change_7d_pct,
    "change_1d_usd": (today_total - yesterday_total) if yesterday_available else None,
    "change_1d_pct": (((today_total / yesterday_total) - 1.0) * 100.0) if (yesterday_available and yesterday_total) else None,
    "change_7d_usd": (today_total - week_ago_total) if week_ago_available else None,
    "change_7d_pct": (((today_total / week_ago_total) - 1.0) * 100.0) if (week_ago_available and week_ago_total) else None,
    "market_breadth": breadth,
    "protocols_tracked": protocols_tracked,
    "top3_details": top3_details,
    "top4_10_share_pct": top4_10_share_pct,
    "near_top10_focus": near_top10_focus,
    "biggest_asset_gainer": biggest_asset_gainer,
    "category_leader_24h": category_leader_24h,
    "top3_concentration": top3_concentration * 100.0 if top3_concentration is not None else None,
    "top10_concentration": top10_concentration * 100.0 if top10_concentration is not None else None,
    "top3_assets": top3_assets,
    "largest_capital_increase_category": None if largest_capital_increase_row.empty else largest_capital_increase_row.iloc[0]["category"],
    "largest_capital_increase_usd": None if largest_capital_increase_row.empty else float(largest_capital_increase_row.iloc[0]["change_usd"]),
    "strongest_category_growth": None if strongest_category_growth_row.empty else strongest_category_growth_row.iloc[0]["category"],
    "strongest_category_growth_pct": None if strongest_category_growth_row.empty else float(strongest_category_growth_row.iloc[0]["change_pct"]),
    "largest_asset_tvl_increase": None if largest_asset_increase_row.empty else largest_asset_increase_row.iloc[0]["name"],
    "largest_asset_tvl_increase_usd": None if largest_asset_increase_row.empty else float(largest_asset_increase_row.iloc[0]["change_1d_usd"]),
    "largest_asset_tvl_decline": None if largest_asset_decline_row.empty else largest_asset_decline_row.iloc[0]["name"],
    "largest_asset_tvl_decline_usd": None if largest_asset_decline_row.empty else float(largest_asset_decline_row.iloc[0]["change_1d_usd"]),
    "new_entrants": int(len(new_entrants_df)),
    "new_top10_entrants": int(len(new_top10_entrants_df)),
}
headline_metrics["market_regime"] = build_market_regime(headline_metrics)

headline_metrics


In [ ]:

# ----------------------------
# SIGNALS / NARRATIVE / TITLE ROTATION
# ----------------------------

TITLE_ROTATION = [
    "Daily RWA Market Recap",
    "RWA Market Brief",
    "Daily RWA Structure Note",
    "Capital Allocation Update",
    "RWA Daily Snapshot & Analysis",
]

def pick_rotating_title(day: date) -> str:
    idx = (day.toordinal()) % len(TITLE_ROTATION)
    return TITLE_ROTATION[idx]

def trend_arrow(x: Optional[float]) -> str:
    if x is None or not np.isfinite(x) or abs(x) < 1e-12:
        return "→"
    return "↑" if x > 0 else "↓"

def build_monitoring_signals(metrics: Dict[str, Any], category_1d_df: pd.DataFrame, today_df: pd.DataFrame) -> List[str]:
    signals = []
    if metrics.get("top3_concentration") is not None:
        signals.append(f"Top 3 concentration stood at {metrics['top3_concentration']:.1f}% of tracked market value.")
    if metrics.get("market_breadth") is not None:
        signals.append(f"Market breadth was {metrics['market_breadth']*100:.1f}% across assets above the $1M threshold.")
    if metrics.get("new_entrants", 0) > 0:
        signals.append(f"{metrics['new_entrants']} asset{'s' if metrics['new_entrants'] != 1 else ''} crossed the $1M threshold in this UTC snapshot.")
    if not category_1d_df.empty:
        top_cat = category_1d_df.sort_values("change_usd", ascending=False).iloc[0]
        signals.append(f"{top_cat['category']} led category-level 24H value change by absolute dollars.")
    if metrics.get("top10_concentration") is not None:
        signals.append(f"Top 10 concentration ended the day at {metrics['top10_concentration']:.1f}% of tracked market value.")
    return signals[:4]

def build_watch_tomorrow(metrics: Dict[str, Any], today_df: pd.DataFrame, yesterday_df: pd.DataFrame, category_1d_df: pd.DataFrame, rank_watch_items: List[Dict[str, Any]]) -> List[str]:
    lines = []

    if metrics.get("top3_concentration") is not None and len(metrics.get("top3_assets", [])) >= 3:
        names = ", ".join(metrics["top3_assets"])
        lines.append(
            f"Capital remained concentrated in {names}. The next snapshot will show whether participation broadens beyond the current leaders."
        )

    if rank_watch_items:
        candidates = ", ".join([item["name"] for item in rank_watch_items[:3]])
        lines.append(
            f"{candidates} remain relevant to the Top 10 threshold, making rank reordering a key point to monitor in the next UTC update."
        )

    if metrics.get("market_breadth") is not None:
        lines.append(
            f"Breadth closed at {metrics['market_breadth']*100:.1f}% today. The next update will show whether positive participation stays above the recent pattern."
        )

    if not category_1d_df.empty:
        leader = category_1d_df.sort_values("change_usd", ascending=False).iloc[0]
        lines.append(
            f"{leader['category']} led category-level value expansion today. Tomorrow's snapshot will confirm whether that move is sustained on a 2-day basis."
        )

    deduped = []
    seen = set()
    for line in lines:
        key = re.sub(r"\s+", " ", line.strip().lower())
        if key not in seen:
            deduped.append(line)
            seen.add(key)
    return deduped[:4]

def build_summary(metrics: Dict[str, Any], today_df: pd.DataFrame, category_1d_df: pd.DataFrame) -> str:
    total = fmt_money_compact_signed(metrics["total_rwa_tvl"])
    ch1d_usd = metrics.get("change_1d_usd")
    ch7d_usd = metrics.get("change_7d_usd")
    breadth = metrics.get("market_breadth")
    top3_assets = ", ".join(metrics.get("top3_assets", [])[:3])
    lead_cat = None if category_1d_df.empty else category_1d_df.sort_values("change_usd", ascending=False).iloc[0]["category"]

    s1 = f"Tracked RWA market value closed at {total} on {date_to_human(SNAPSHOT_DATE)} (UTC)."
    if ch1d_usd is not None:
        direction = "increased" if ch1d_usd > 0 else "decreased" if ch1d_usd < 0 else "was unchanged"
        if direction == "was unchanged":
            s2 = "On a 24-hour basis, aggregate value was unchanged."
        else:
            s2 = f"On a 24-hour basis, aggregate value {direction} by {fmt_money_compact_signed(abs(ch1d_usd))}."
    else:
        s2 = "A 24-hour comparison was not available in local history."
    if ch7d_usd is not None:
        direction7 = "higher" if ch7d_usd > 0 else "lower" if ch7d_usd < 0 else "flat"
        s3 = f"Against the local 7-day history, market value was {direction7} by {fmt_money_compact_signed(abs(ch7d_usd))}."
    else:
        s3 = "A 7-day comparison becomes available once the local analytics history reaches seven UTC snapshots."
    if breadth is not None and lead_cat:
        s4 = f"{lead_cat} led category-level movement, while market breadth was {breadth*100:.1f}% across eligible assets. The largest share remained concentrated in {top3_assets}."
    elif breadth is not None:
        s4 = f"Market breadth was {breadth*100:.1f}% across eligible assets, while the largest share remained concentrated in {top3_assets}."
    else:
        s4 = f"The largest share remained concentrated in {top3_assets}."
    return " ".join([s1, s2, s3, s4])

def build_x_share_text(metrics: Dict[str, Any], payload_title: str, category_1d_df: pd.DataFrame, page_path: str) -> str:
    lead_cat = None if category_1d_df.empty else category_1d_df.sort_values("change_usd", ascending=False).iloc[0]["category"]
    parts = [
        payload_title,
        f"Total RWA TVL: {fmt_money_compact_signed(metrics.get('total_rwa_tvl'))}",
        f"24H Change: {fmt_pct(metrics.get('change_1d_pct'))}",
        f"7D Change: {fmt_pct(metrics.get('change_7d_pct'))}",
        f"Market Breadth: {'N/A' if metrics.get('market_breadth') is None else f"{metrics['market_breadth']*100:.1f}%"}",
    ]
    if lead_cat:
        parts.append(f"Largest capital increase: {lead_cat}")
    parts.append(page_path)
    return " | ".join(parts)

monitoring_signals = build_monitoring_signals(headline_metrics, category_1d_df, today_df)
watch_tomorrow = build_watch_tomorrow(headline_metrics, today_df, yesterday_df, category_1d_df, rank_watch_items)
article_title = pick_rotating_title(SNAPSHOT_DATE)
hero_summary = build_summary(headline_metrics, today_df, category_1d_df)


In [ ]:
# ----------------------------
# CHART RENDERING
# ----------------------------

def load_total_series() -> pd.DataFrame:
    rows = []
    for day, snap in all_snapshots.items():
        total = 0.0
        for asset in snap.get("assets", []):
            total += safe_float(asset.get("tvl_usd")) or 0.0
        rows.append({"date": day, "total_rwa_tvl": total})
    df = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
    return df

def render_total_7d_chart(series_df: pd.DataFrame, out_path: Path, title: str = "Total RWA TVL — Last 7 Days") -> None:
    plot_df = series_df.copy()
    if plot_df.empty:
        plot_df = pd.DataFrame([{"date": SNAPSHOT_DATE_STR, "total_rwa_tvl": 0.0}])

    plot_df["date_obj"] = pd.to_datetime(plot_df["date"])
    plot_df = plot_df.sort_values("date_obj").tail(7)

    fig, ax = plt.subplots(figsize=(12, 5.5), dpi=160)
    bg = "#0b1220"
    fig.patch.set_facecolor(bg)
    ax.set_facecolor(bg)

    x = np.arange(len(plot_df))
    y = plot_df["total_rwa_tvl"].astype(float).values

    # blue glow layers
    for lw, alpha in [(12, 0.08), (8, 0.12), (5, 0.18)]:
        ax.plot(x, y, linewidth=lw, alpha=alpha, color="#5ED6FF", solid_capstyle="round")

    ax.plot(x, y, linewidth=2.8, color="#5ED6FF", solid_capstyle="round")
    ax.fill_between(x, y, y2=min(y) if len(y) else 0.0, color="#4DA3FF", alpha=0.10)

    ax.grid(True, axis="y", linestyle="-", linewidth=0.8, alpha=0.10, color="#9fb3c8")
    ax.grid(True, axis="x", linestyle="-", linewidth=0.6, alpha=0.06, color="#9fb3c8")

    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.tick_params(colors="#c7d2e0", labelsize=10)
    labels = [pd.to_datetime(d).strftime("%b %d") for d in plot_df["date_obj"]]
    ax.set_xticks(x)
    ax.set_xticklabels(labels, color="#c7d2e0")
    ax.yaxis.set_major_formatter(FuncFormatter(lambda val, pos: fmt_money_compact_abs(val)))
    for label in ax.get_yticklabels():
        label.set_color("#c7d2e0")

    ax.set_title(title, color="white", fontsize=16, fontweight="bold", pad=16)
    ax.text(
        0.995, 0.04, "RWA.watch",
        transform=ax.transAxes,
        ha="right", va="bottom",
        fontsize=16, fontweight="bold",
        color=(1, 1, 1, 0.13)
    )

    fig.tight_layout()
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, facecolor=fig.get_facecolor(), bbox_inches="tight")
    plt.close(fig)

total_series_df = load_total_series()
daily_chart_path = CHARTS_ROOT / f"total-rwa-tvl-7d-{SNAPSHOT_DATE_STR}.png"
render_total_7d_chart(total_series_df, daily_chart_path)

# public copy
site_daily_chart_path = SITE_CHARTS_ROOT / daily_chart_path.name
shutil.copy2(daily_chart_path, site_daily_chart_path)

display(total_series_df.tail(10))
print("Chart:", daily_chart_path)


In [ ]:

# ----------------------------
# ARTICLE JSON / PAGE DATA
# ----------------------------

def build_category_cards(category_1d_df: pd.DataFrame, category_7d_df: pd.DataFrame) -> List[Dict[str, Any]]:
    out = []
    cat7_map = {row["category"]: row.to_dict() for _, row in category_7d_df.iterrows()} if not category_7d_df.empty else {}
    for _, row in category_1d_df.iterrows():
        row7 = cat7_map.get(row["category"])
        change_7d_usd = None if row7 is None else safe_float(row7.get("change_usd"))
        change_7d_pct = None if row7 is None else safe_float(row7.get("change_pct"))
        out.append({
            "category": row["category"],
            "tvl_usd": float(row["tvl_usd"]),
            "share": row["share"],
            "change_1d_usd": safe_float(row.get("change_usd")),
            "change_1d_pct": safe_float(row.get("change_pct")),
            "change_7d_usd": change_7d_usd,
            "change_7d_pct": change_7d_pct,
            "trend_arrow": trend_arrow(safe_float(row.get("change_pct"))),
        })
    return out

def build_asset_leadership(today_df: pd.DataFrame) -> Dict[str, Any]:
    elig = today_df.loc[today_df["eligible_today"]].copy()
    if elig.empty:
        elig = today_df.copy()
    increase = elig.sort_values("change_1d_usd", ascending=False).head(5)
    decline = elig.sort_values("change_1d_usd", ascending=True).head(5)

    return {
        "largest_increase": increase[["name", "slug", "analytics_category", "tvl_usd", "change_1d_usd", "change_1d_pct"]].to_dict("records"),
        "largest_decline": decline[["name", "slug", "analytics_category", "tvl_usd", "change_1d_usd", "change_1d_pct"]].to_dict("records"),
        "rank_watch": rank_watch_items,
    }

daily_permalink = f"/analysis/recap-{SNAPSHOT_DATE_STR}.html"
x_share_text = build_x_share_text(headline_metrics, article_title, category_1d_df, daily_permalink)

daily_page_payload = {
    "title": article_title,
    "date": SNAPSHOT_DATE_STR,
    "display_date": date_to_human(SNAPSHOT_DATE),
    "summary": hero_summary,
    "permalink": daily_permalink,
    "x_share_text": x_share_text,
    "market_regime": headline_metrics.get("market_regime"),
    "main_chart": {
        "title": "Total RWA TVL — Last 7 Days",
        "image": f"charts/{daily_chart_path.name}",
    },
    "headline_metrics": headline_metrics,
    "what_to_watch_tomorrow": watch_tomorrow,
    "capital_allocation": build_category_cards(category_1d_df, category_7d_df),
    "market_structure": {
        "top3_concentration_pct": headline_metrics["top3_concentration"],
        "top10_concentration_pct": headline_metrics["top10_concentration"],
        "top3_assets": headline_metrics["top3_assets"],
        "largest_assets": today_df.head(10)[["name", "slug", "analytics_category", "tvl_usd", "share"]].to_dict("records"),
    },
    "asset_leadership": build_asset_leadership(today_df),
    "category_trends": [
        {"category": row["category"], "arrow": row["trend_arrow"], "change_1d_pct": row["change_1d_pct"], "change_1d_usd": row["change_1d_usd"]}
        for row in build_category_cards(category_1d_df, category_7d_df)[:5]
    ],
    "monitoring_signals": monitoring_signals,
    "new_entrants": new_entrants_df[["name", "slug", "analytics_category", "tvl_usd", "rank_by_tvl"]].head(10).to_dict("records"),
}

write_json(SITE_DATA_DAILY_ROOT / f"{SNAPSHOT_DATE_STR}.json", daily_page_payload)
write_json(SITE_DATA_METRICS_ROOT / f"headline-metrics-{SNAPSHOT_DATE_STR}.json", headline_metrics)


In [ ]:

# ----------------------------
# PRIVATE CSV EXPORTS
# ----------------------------

def build_market_metrics_csv(total_series_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    available_dates = sorted(all_snapshots.keys())
    for d in available_dates:
        current = get_day_df(d)
        prev = get_day_df((date.fromisoformat(d) - timedelta(days=1)).isoformat())
        top3c = None
        top10c = None
        if not current.empty and current["tvl_usd"].sum():
            denom = float(current["tvl_usd"].sum())
            top3c = (current.head(3)["tvl_usd"].sum() / denom)
            top10c = (current.head(10)["tvl_usd"].sum() / denom)

        top500_now = float(current.head(500)["tvl_usd"].sum()) if not current.empty else 0.0
        top500_prev = float(prev.head(500)["tvl_usd"].sum()) if not prev.empty else None
        eligible = current.loc[current["tvl_usd"] >= ELIGIBLE_THRESHOLD_USD].copy()
        if not eligible.empty:
            prev_map = keyed(prev)
            eligible["delta"] = eligible["slug"].map(lambda s: eligible.loc[eligible["slug"] == s, "tvl_usd"].iloc[0] - safe_float(prev_map.get(s, {}).get("tvl_usd") or 0.0))
            breadth_val = float((eligible["delta"] > 0).sum() / len(eligible))
        else:
            breadth_val = None

        total_now = float(current["tvl_usd"].sum()) if not current.empty else 0.0
        total_prev = float(prev["tvl_usd"].sum()) if not prev.empty else None
        rows.append({
            "date": d,
            "total_rwa_tvl": total_now,
            "change_1d_usd": None if total_prev is None else (total_now - total_prev),
            # NOTE: pct columns here are ratios (0.05 == +5%). Use *_pct_points for %-points.
            "change_1d_pct": None if not total_prev else ((total_now / total_prev) - 1.0),
            "change_1d_pct_points": None if not total_prev else (((total_now / total_prev) - 1.0) * 100.0),
            "breadth": breadth_val,
            "top3_concentration": top3c,
            "top10_concentration": top10c,
            "top500_rwa_tvl": top500_now,
            "top500_change_1d_usd": None if top500_prev is None else (top500_now - top500_prev),
            "top500_change_1d_pct": None if not top500_prev else ((top500_now / top500_prev) - 1.0),
            "top500_change_1d_pct_points": None if not top500_prev else (((top500_now / top500_prev) - 1.0) * 100.0),
        })
    return pd.DataFrame(rows).sort_values("date")

def build_category_timeseries_csv() -> pd.DataFrame:
    rows = []
    for d in sorted(all_snapshots.keys()):
        current = get_day_df(d)
        prev = get_day_df((date.fromisoformat(d) - timedelta(days=1)).isoformat())
        cats = sorted(set(current["analytics_category"].tolist()) | set(prev["analytics_category"].tolist()))
        total = float(current["tvl_usd"].sum()) if not current.empty else 0.0
        for cat in cats:
            tvl_now = float(current.loc[current["analytics_category"] == cat, "tvl_usd"].sum())
            tvl_prev = float(prev.loc[prev["analytics_category"] == cat, "tvl_usd"].sum())
            rows.append({
                "date": d,
                "category": cat,
                "tvl_usd": tvl_now,
                "share": None if total == 0 else tvl_now / total,
                "change_1d_usd": tvl_now - tvl_prev,
                "change_1d_pct": None if tvl_prev == 0 else ((tvl_now / tvl_prev) - 1.0),
            })
    return pd.DataFrame(rows).sort_values(["date", "tvl_usd"], ascending=[True, False])

def build_asset_timeseries_csv() -> pd.DataFrame:
    rows = []
    for d in sorted(all_snapshots.keys()):
        current = get_day_df(d)
        prev = get_day_df((date.fromisoformat(d) - timedelta(days=1)).isoformat())
        prev_map = keyed(prev)
        total = float(current["tvl_usd"].sum()) if not current.empty else 0.0
        for _, row in current.iterrows():
            prev_tvl = safe_float(prev_map.get(row["slug"], {}).get("tvl_usd"))
            is_new_since_yesterday = prev_tvl is None
            rows.append({
                "date": d,
                "asset_id": row.get("asset_id") or row["slug"],
                "asset_key": row.get("asset_key") or row.get("asset_id") or row["slug"],
                "slug": row["slug"],
                "name": row["name"],
                "issuer": row.get("issuer"),
                "issuer_id": row.get("issuer_id"),
                "asset_type": row.get("asset_type"),
                "primary_chain": row.get("primary_chain"),
                "chains": json.dumps(row.get("chains") or []),
                "category": row["analytics_category"],
                "tvl_usd": float(row["tvl_usd"]),
                "rank_by_tvl": int(row["rank_by_tvl"]),
                "share": None if total == 0 else float(row["tvl_usd"]) / total,
                "prev_tvl_usd": prev_tvl,
                "change_1d_usd": None if is_new_since_yesterday else (float(row["tvl_usd"]) - float(prev_tvl)),
                "change_1d_pct": None if (is_new_since_yesterday or float(prev_tvl) == 0) else ((float(row["tvl_usd"]) / float(prev_tvl)) - 1.0),
                "in_top500_today": bool((int(row["rank_by_tvl"]) <= 500) and (float(row["tvl_usd"]) >= ELIGIBLE_THRESHOLD_USD)),
                "is_new_since_yesterday": bool(is_new_since_yesterday),
            })
    return pd.DataFrame(rows).sort_values(["date", "rank_by_tvl"])


def build_gold_asset_state_csv() -> pd.DataFrame:
    """Minimal 'gold' dataset: date, asset_id/key, tvl_usd, rank, category (+ a few durable dims)."""
    rows = []
    for d in sorted(all_snapshots.keys()):
        current = get_day_df(d)
        if current.empty:
            continue
        for _, row in current.iterrows():
            rows.append({
                "date": d,
                "asset_id": row.get("asset_id") or row["slug"],
                "asset_key": row.get("asset_key") or row.get("asset_id") or row["slug"],
                "slug": row.get("slug"),
                "name": row.get("name"),
                "tvl_usd": float(row.get("tvl_usd") or 0.0),
                "rank_by_tvl": int(row.get("rank_by_tvl") or 0),
                "category": row.get("analytics_category") or row.get("category") or "Other RWA",
                "asset_type": row.get("asset_type"),
                "issuer": row.get("issuer"),
                "issuer_id": row.get("issuer_id"),
                "primary_chain": row.get("primary_chain"),
                "chains": json.dumps(row.get("chains") or []),
            })
    return pd.DataFrame(rows).sort_values(["date", "rank_by_tvl"])

def build_daily_market_summary_csv(metrics: Dict[str, Any]) -> pd.DataFrame:
    rows = [
        {"metric": "Date", "value": SNAPSHOT_DATE_STR},
        {"metric": "Total RWA TVL", "value": metrics.get("total_rwa_tvl")},
        {"metric": "Top 500 RWA TVL", "value": metrics.get("top500_rwa_tvl")},
        {"metric": "Top 500 24H Change USD", "value": metrics.get("top500_change_1d_usd")},
        {"metric": "Top 500 24H Change %", "value": metrics.get("top500_change_1d_pct")},
        {"metric": "Top 500 7D Change USD", "value": metrics.get("top500_change_7d_usd")},
        {"metric": "Top 500 7D Change %", "value": metrics.get("top500_change_7d_pct")},
        {"metric": "24H Change USD", "value": metrics.get("change_1d_usd")},
        {"metric": "24H Change %", "value": metrics.get("change_1d_pct")},
        {"metric": "7D Change USD", "value": metrics.get("change_7d_usd")},
        {"metric": "7D Change %", "value": metrics.get("change_7d_pct")},
        {"metric": "Market Breadth (experimental)", "value": metrics.get("market_breadth")},
        {"metric": "Top 3 Concentration", "value": metrics.get("top3_concentration")},
        {"metric": "Top 10 Concentration", "value": metrics.get("top10_concentration")},
        {"metric": "Largest Capital Increase Category", "value": metrics.get("largest_capital_increase_category")},
        {"metric": "Strongest Category Growth", "value": metrics.get("strongest_category_growth")},
        {"metric": "New Entrants", "value": metrics.get("new_entrants")},
        {"metric": "New Top 10 Entrants", "value": metrics.get("new_top10_entrants")},
    ]
    return pd.DataFrame(rows)

def build_daily_category_breakdown_csv(category_cards: List[Dict[str, Any]]) -> pd.DataFrame:
    return pd.DataFrame(category_cards)

def build_daily_top_assets_csv(today_df: pd.DataFrame) -> pd.DataFrame:
    cols = ["rank_by_tvl", "name", "slug", "analytics_category", "tvl_usd", "share", "change_1d_usd", "change_7d_usd"]
    out = today_df[cols].head(25).copy()
    out = out.rename(columns={"analytics_category": "category"})
    return out


def build_daily_top500_assets_csv(today_df: pd.DataFrame, yesterday_df: pd.DataFrame) -> pd.DataFrame:
    # Investor-facing: per-asset TVL + 24H change for the current Top 500 (eligible) assets.
    prev_map = keyed(yesterday_df)
    total = float(today_df["tvl_usd"].sum()) if not today_df.empty else 0.0

    rows = []
    top500 = today_df.loc[today_df["in_top500_today"]].sort_values(["rank_by_tvl", "slug"])
    for _, row in top500.iterrows():
        prev = prev_map.get(row["slug"])
        prev_tvl = safe_float(prev.get("tvl_usd")) if prev else None
        tvl = float(row["tvl_usd"])
        rows.append({
            "date": SNAPSHOT_DATE_STR,
            "rank_by_tvl": int(row["rank_by_tvl"]),
            "name": row["name"],
            "slug": row["slug"],
            "category": row["analytics_category"],
            "tvl_usd": tvl,
            "share": None if total == 0 else tvl / total,
            "change_1d_usd": None if prev_tvl is None else (tvl - float(prev_tvl)),
            "change_1d_pct": None if (prev_tvl is None or float(prev_tvl) == 0) else ((tvl / float(prev_tvl)) - 1.0),
            "is_new_since_yesterday": bool(prev_tvl is None),
        })

    return pd.DataFrame(rows)


def build_daily_rank_watch_csv(rank_watch_items: List[Dict[str, Any]]) -> pd.DataFrame:
    return pd.DataFrame(rank_watch_items)

def build_daily_new_entrants_csv(new_entrants_df: pd.DataFrame) -> pd.DataFrame:
    if new_entrants_df.empty:
        return pd.DataFrame(columns=["name", "slug", "analytics_category", "tvl_usd", "rank_by_tvl"])
    return new_entrants_df[["name", "slug", "analytics_category", "tvl_usd", "rank_by_tvl"]].rename(columns={"analytics_category": "category"})

market_metrics_csv_df = build_market_metrics_csv(total_series_df)
category_timeseries_csv_df = build_category_timeseries_csv()
asset_timeseries_csv_df = build_asset_timeseries_csv()

gold_asset_state_csv_df = build_gold_asset_state_csv()

market_metrics_path = EXPORTS_ROOT / "market_metrics.csv"
category_timeseries_path = EXPORTS_ROOT / "category_timeseries.csv"
asset_timeseries_path = EXPORTS_ROOT / "asset_timeseries.csv"
gold_asset_state_path = EXPORTS_ROOT / "gold_asset_state.csv"

market_metrics_csv_df.to_csv(market_metrics_path, index=False)
category_timeseries_csv_df.to_csv(category_timeseries_path, index=False)
asset_timeseries_csv_df.to_csv(asset_timeseries_path, index=False)
gold_asset_state_csv_df.to_csv(gold_asset_state_path, index=False)

investor_daily_market_summary_df = build_daily_market_summary_csv(headline_metrics)
investor_daily_category_breakdown_df = build_daily_category_breakdown_csv(build_category_cards(category_1d_df, category_7d_df))
investor_daily_top_assets_df = build_daily_top_assets_csv(today_df)
investor_daily_top500_assets_df = build_daily_top500_assets_csv(today_df, yesterday_df)
investor_daily_rank_watch_df = build_daily_rank_watch_csv(rank_watch_items)
investor_daily_new_entrants_df = build_daily_new_entrants_csv(new_entrants_df)

investor_daily_market_summary_df.to_csv(EXPORTS_ROOT / "daily_market_summary.csv", index=False)
investor_daily_category_breakdown_df.to_csv(EXPORTS_ROOT / "daily_category_breakdown.csv", index=False)
investor_daily_top_assets_df.to_csv(EXPORTS_ROOT / "daily_top_assets.csv", index=False)
investor_daily_top500_assets_df.to_csv(EXPORTS_ROOT / "daily_top500_assets.csv", index=False)
investor_daily_rank_watch_df.to_csv(EXPORTS_ROOT / "daily_rank_watch.csv", index=False)
investor_daily_new_entrants_df.to_csv(EXPORTS_ROOT / "daily_new_entrants.csv", index=False)

(SITE_EXPORTS_ROOT / "README.txt").write_text(
    "Investor-grade CSV exports are generated in the private analytics namespace and are not published on the public site.",
    encoding="utf-8"
)

print("Private CSV exports saved to:", EXPORTS_ROOT)
display(investor_daily_market_summary_df.head(15))


In [ ]:

# ----------------------------
# WEEKLY SUMMARY
# ----------------------------

def dates_in_week(year: int, week: int) -> List[str]:
    candidates = []
    for d in sorted(all_snapshots.keys()):
        dd = date.fromisoformat(d)
        y, w, _ = dd.isocalendar()
        if y == year and w == week:
            candidates.append(d)
    return candidates

week_dates = dates_in_week(ISO_YEAR, ISO_WEEK)
week_snapshots = [all_snapshots[d] for d in week_dates]

weekly_payload = {
    "title": f"Weekly RWA Market Recap — Week {ISO_WEEK}, {ISO_YEAR}",
    "week_label": WEEK_LABEL,
    "display_week": f"Week {ISO_WEEK}, {ISO_YEAR}",
    "dates_included": week_dates,
    "summary": (
        f"This weekly recap aggregates the UTC snapshots available for {WEEK_LABEL}. "
        f"It tracks market value, investor-facing category shifts, concentration, asset-level changes, and rank pressure using the analytics namespace."
    ),
    "headline_metrics": headline_metrics,
    "category_shifts": build_category_cards(category_1d_df, category_7d_df),
    "concentration_change": {
        "top3_concentration_pct": headline_metrics["top3_concentration"],
        "top10_concentration_pct": headline_metrics["top10_concentration"],
    },
    "largest_asset_change": build_asset_leadership(today_df),
    "new_entrants": new_entrants_df[["name", "slug", "analytics_category", "tvl_usd", "rank_by_tvl"]].head(10).to_dict("records"),
    "what_to_watch_next_week": watch_tomorrow,
}
write_json(SITE_DATA_WEEKLY_ROOT / f"{WEEK_LABEL}.json", weekly_payload)
weekly_payload["title"]


In [ ]:

# ----------------------------
# SHARED SHELL (embedded from the main-site reference)
# ----------------------------

SHELL_CSS = r'''
:root{
  --bg:#010409; --panel:#11161d; --panel2:#151b23; --border:rgba(255,255,255,0.10);
  --text:rgba(255,255,255,0.86); --muted:rgba(255,255,255,0.62); --link:#7ab7ff;
  --betaBg:#1d4ed8; --betaBorder:#1e40af; --blue:#4da3ff;
}
*{box-sizing:border-box;}
html,body{height:100%;}
body{
  margin:0;
  background:#0b1220 !important;
  background-image:none !important;
  color:var(--text);
  font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",Inter,Helvetica,Arial;
}
a{color:var(--link);text-decoration:none;}
a:hover{text-decoration:underline;}


.container{max-width:1200px;margin:0 auto;padding:28px 20px 40px;}
.topbar{display:flex;align-items:flex-start;justify-content:space-between;gap:14px;flex-wrap:wrap;}
.brand{display:flex;align-items:center;gap:12px;}
.brand h1{margin:0;color:#fff;font-size:34px;letter-spacing:0.2px;}
.beta{font-size:12px;padding:6px 10px;border-radius:999px;background:var(--betaBg);border:1px solid var(--betaBorder);color:#fff;font-weight:900;}
.updated{color:var(--muted);font-size:12px;margin-top:8px;}
.headline{margin-top:14px;}
.headline h2{margin:0;color:#fff;font-size:22px;}
.headline p{margin:10px 0 0 0;color:var(--muted);line-height:1.7;max-width:980px;}
.tabs{margin-top:18px;display:flex;gap:10px;flex-wrap:wrap;}
.tab{
  border:1px solid var(--border);
  background: rgba(255,255,255,0.03);
  padding:10px 14px;border-radius:12px;
  font-weight:900;color:var(--text);
  backdrop-filter: blur(8px);
}
.tab.active{border-color: rgba(122,183,255,0.8); background: rgba(122,183,255,0.06);}
.footer{margin-top:32px;color:var(--muted);font-size:12px;text-align:center;line-height:1.7;}
.legalLine{display:inline;}
.legalDetails{display:inline-block; margin-left:6px;}
.legalDetails summary{cursor:pointer; display:inline; color:var(--link);} 
.legalDetails[open] summary{opacity:0.9;}
.legalBody{margin-top:10px; text-align:left; max-width:820px; margin-left:auto; margin-right:auto; padding:12px; border:1px solid var(--border); border-radius:12px; background:rgba(255,255,255,0.03);}
.legalBody p{margin:0 0 8px 0; color:var(--muted);} 
.copyright{margin-top:6px;}
:root{
  --posBlue:#5ED6FF; --negBlue:#4DA3FF; --posClassic:#5EE6A8; --negClassic:#FF6B7A;
  --pos:var(--posBlue); --neg:var(--negBlue); --zero:#FFFFFF;
}
html[data-delta-mode="classic"]{ --pos:var(--posClassic); --neg:var(--negClassic); }
.chg{font-weight:900;}
.chg.pos{color:var(--pos)!important;text-shadow:0 0 10px rgba(94,214,255,0.34);}
.chg.neg{color:var(--neg)!important;text-shadow:0 0 8px rgba(255,107,122,0.20);}
.chg.zero{color:var(--zero)!important;}
.chg.na{color:var(--zero)!important;font-weight:800;}
.modeToggle{display:inline-flex;gap:6px;align-items:center;margin-left:10px;flex-wrap:wrap;}
.modeBtn{
  border:1px solid var(--border); background:rgba(0,0,0,0.18); color:#fff; padding:6px 10px;
  border-radius:999px; cursor:pointer; font-weight:900; font-size:12px; line-height:1;
}
.modeBtn.active{border-color:var(--link);}

/* --- Compact legal footer (shared) --- */
.footer{margin-top:24px;color:var(--muted);font-size:12px;text-align:center;line-height:1.7;}
.footLine{max-width:980px;margin:0 auto;}
.legal{margin-top:10px;display:inline-block;text-align:left;max-width:980px;}
.legal summary{cursor:pointer;color:var(--link);font-weight:900;outline:none;}
.legal summary::-webkit-details-marker{display:none;}
.legalBody{margin-top:8px;color:var(--muted);line-height:1.7;}

/* --- Mobile header fixes (shared) --- */
@media (max-width: 520px){
  .topbar{flex-direction:column;align-items:flex-start;}
  .brand{flex-wrap:wrap;row-gap:8px;}
  .modeToggle{margin-left:0;}
  .updated{margin-top:6px;}
  .tabs{flex-wrap:nowrap;overflow-x:auto;-webkit-overflow-scrolling:touch;padding-bottom:6px;}
  .tabs::-webkit-scrollbar{display:none;}
  .tab{flex:0 0 auto;}
}
'''

ANALYTICS_CSS = r'''
.heroShell,.sectionCard,.metricCard,.rowCard,.chartCard,.highlightCard,.textPanel,.miniPanel{
  position:relative;
  overflow:hidden;
  isolation:isolate;
  background: linear-gradient(180deg, rgba(255,255,255,0.040), rgba(255,255,255,0.022));
  border:1px solid var(--border);
  border-radius:20px;
  backdrop-filter: blur(12px);
  box-shadow:
    inset 0 0 0 1px rgba(255,255,255,0.012),
    0 12px 34px rgba(0,0,0,0.18);
}
.heroShell::before,.sectionCard::before,.metricCard::before,.rowCard::before,.miniPanel::before{
  content:"";
  position:absolute;
  inset:-20% auto auto -20%;
  width:52%;
  height:52%;
  border-radius:999px;
  background:radial-gradient(circle, rgba(94,214,255,0.12), rgba(94,214,255,0.00) 70%);
  pointer-events:none;
  z-index:0;
}
.heroShell > *, .sectionCard > *, .metricCard > *, .rowCard > *, .miniPanel > *{position:relative;z-index:1;}

.heroShell{margin-top:18px;padding:22px;}
.heroTop{display:grid;grid-template-columns:1.02fr 1.18fr;gap:18px;align-items:stretch;}
@media (max-width: 980px){ .heroTop{grid-template-columns:1fr;} }

.heroEyebrow{color:var(--muted);font-size:12px;letter-spacing:0.10em;text-transform:uppercase;}
.heroTitle{margin:8px 0 10px 0;color:#fff;font-size:30px;line-height:1.03;letter-spacing:-0.02em;}
.heroDate{color:var(--muted);font-size:13px;}
.heroSummary{margin:14px 0 0 0;color:var(--text);line-height:1.75;max-width:680px;font-size:15px;}
.heroLeft{display:flex;flex-direction:column;justify-content:space-between;gap:18px;}
.heroChartWrap{position:relative;border-radius:18px;overflow:hidden;border:1px solid rgba(94,214,255,0.10);background:#07111b;min-height:100%;}
.heroChartWrap img{display:block;width:100%;height:auto;}
.heroChartLabel{
  position:absolute;right:18px;bottom:18px;padding:10px 14px;border-radius:14px;
  background:rgba(6,10,18,0.84);border:1px solid var(--border);font-size:14px;font-weight:900;color:#fff;
  box-shadow:0 0 24px rgba(77,163,255,0.18);
}
.heroChartCaption{position:absolute;left:18px;top:18px;padding:8px 12px;border-radius:999px;background:rgba(6,10,18,0.84);border:1px solid var(--border);font-size:11px;color:var(--muted);}
.kpiStrip{margin-top:16px;display:flex;align-items:center;gap:10px;flex-wrap:wrap;}
.kpiStrip .badgePill{font-size:12px;}
.marketRegime{
  margin-top:16px;padding:14px 16px;border-radius:16px;border:1px solid rgba(94,214,255,0.16);
  background:linear-gradient(90deg, rgba(22,87,160,0.24), rgba(7,18,35,0.18));
  color:#d9edff;font-size:14px;font-weight:700;
}
.quickNav{display:flex;gap:10px;flex-wrap:wrap;margin-top:16px;}
.quickNav a{
  display:inline-flex;align-items:center;gap:8px;padding:10px 12px;border-radius:12px;
  border:1px solid var(--border);background:rgba(4,11,18,0.46);color:#fff;font-size:12px;font-weight:800;
}
.quickNav a:hover{text-decoration:none;border-color:rgba(122,183,255,0.7);}

.metricGridPrimary{margin-top:18px;display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:14px;}
.metricGridSecondary{margin-top:14px;display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:14px;}
@media (max-width: 1100px){ .metricGridPrimary{grid-template-columns:repeat(2,minmax(0,1fr));} .metricGridSecondary{grid-template-columns:repeat(2,minmax(0,1fr));}}
@media (max-width: 640px){ .metricGridPrimary,.metricGridSecondary{grid-template-columns:1fr;} }

.metricCard{padding:18px;}
.metricCard.primary{padding:20px;min-height:148px;background:linear-gradient(180deg, rgba(18,72,136,0.22), rgba(255,255,255,0.02));}
.metricCard.secondary{min-height:132px;}
.metricCardLabel{color:var(--muted);font-size:12px;letter-spacing:0.05em;text-transform:uppercase;}
.metricCardValue{margin-top:10px;color:#fff;font-size:28px;font-weight:900;letter-spacing:-0.02em;line-height:1.08;}
.metricCard.primary .metricCardValue{font-size:34px;}
.metricCardSub{margin-top:10px;color:var(--muted);font-size:12px;line-height:1.55;}
.metricCardMini{margin-top:12px;font-size:12px;color:#dce7f5;}
.miniRow{display:flex;justify-content:space-between;gap:12px;font-size:12px;color:var(--muted);line-height:1.35;margin-top:4px;}
.miniRow b{color:#dbe7ff;font-weight:600;}

.sectionGrid{margin-top:18px;display:grid;grid-template-columns:1.10fr 0.90fr;gap:16px;}
.sectionGrid.equal{grid-template-columns:1fr 1fr;}
@media (max-width: 980px){ .sectionGrid,.sectionGrid.equal{grid-template-columns:1fr;} }

.sectionCard{padding:18px 18px 16px 18px;}
.sectionCard h3{margin:0;color:#fff;font-size:18px;letter-spacing:-0.01em;}
.sectionCard p.sub{margin:8px 0 0 0;color:var(--muted);font-size:13px;line-height:1.6;}

.inlineList{margin-top:14px;display:flex;flex-direction:column;gap:10px;}
.rowCard{padding:12px 14px;border-radius:16px;background:rgba(0,0,0,0.18);}
.rowTitle{display:flex;align-items:center;justify-content:space-between;gap:10px;flex-wrap:wrap;}
.rowTitle b{color:#fff;}
.rowMeta{margin-top:6px;color:var(--muted);font-size:12px;line-height:1.55;}

.badgePill{
  display:inline-flex;align-items:center;gap:6px;padding:6px 11px;border-radius:999px;border:1px solid var(--border);
  background:rgba(0,0,0,0.18);color:var(--muted);font-size:11px;
}
.badgeWrap{display:flex;gap:8px;flex-wrap:wrap;margin-top:12px;}

.watchList,.signalList,.recapList{margin-top:14px;display:flex;flex-direction:column;gap:10px;}
.watchItem,.signalItem,.recapItem{
  padding:12px 14px;border-radius:14px;border:1px solid var(--border);background:rgba(0,0,0,0.18);line-height:1.7;color:var(--text);
}
.signalItem b, .watchItem b{color:#fff;}
.shareActions{margin-top:14px;display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:10px;}
@media (max-width:640px){ .shareActions{grid-template-columns:1fr;} }
.shareBtn{
  display:flex;align-items:center;justify-content:center;padding:12px 14px;border-radius:14px;border:1px solid var(--border);
  background:rgba(8,13,24,0.70);color:#fff;font-weight:800;text-decoration:none;
}
.shareBtn:hover{text-decoration:none;border-color:rgba(122,183,255,0.7);}

.miniTable{width:100%;border-collapse:collapse;margin-top:14px;}
.miniTable th,.miniTable td{padding:11px 0;border-bottom:1px solid rgba(255,255,255,0.06);text-align:left;font-size:13px;vertical-align:top;}
.miniTable th{color:var(--muted);font-size:12px;font-weight:700;}
.miniTable td{color:var(--text);}
.miniTable tr:last-child td{border-bottom:none;}

.previewBox{margin-top:12px;padding:12px 13px;border:1px dashed var(--border);border-radius:14px;color:var(--muted);font-size:13px;line-height:1.6;}
.copyBox{margin-top:10px;padding:12px;border-radius:12px;background:rgba(0,0,0,0.18);border:1px solid var(--border);color:var(--muted);font-size:12px;line-height:1.6;word-break:break-word;}
.archiveNote{margin-top:8px;color:var(--muted);font-size:12px;line-height:1.6;}

.categoryBar{
  display:grid;grid-template-columns:150px 1fr auto;gap:12px;align-items:center;padding:12px 0;
  border-bottom:1px solid rgba(255,255,255,0.06);
}
.categoryBar:last-child{border-bottom:none;}
.barTrack{height:10px;border-radius:999px;background:rgba(255,255,255,0.06);overflow:hidden;}
.barFill{height:100%;border-radius:999px;background:linear-gradient(90deg, rgba(94,214,255,0.92), rgba(94,214,255,0.28));box-shadow:0 0 18px rgba(94,214,255,0.35);}
.dim{color:var(--muted);font-size:12px;}

.dualPanels{display:grid;grid-template-columns:1fr 1fr;gap:12px;margin-top:14px;}
@media (max-width: 740px){ .dualPanels{grid-template-columns:1fr;} }
.miniPanel{padding:14px;border-radius:16px;background:rgba(0,0,0,0.18);}
.miniPanel h4{margin:0 0 10px 0;font-size:14px;color:#fff;}
.smallMuted{color:var(--muted);font-size:12px;line-height:1.55;}

.recentRow{display:grid;grid-template-columns:1fr 1fr;gap:16px;margin-top:18px;}
@media (max-width:980px){ .recentRow{grid-template-columns:1fr;} }
'''

TOGGLE_SCRIPT = r'''
<script id="delta-toggle-script">
(function(){
  const root = document.documentElement;
  const KEY = "rwa.watch.delta.mode";
  const btnBlue = document.getElementById("modeBlue");
  const btnClassic = document.getElementById("modeClassic");
  function apply(mode){
    root.setAttribute("data-delta-mode", mode === "classic" ? "classic" : "blue");
    if(btnBlue) btnBlue.classList.toggle("active", mode !== "classic");
    if(btnClassic) btnClassic.classList.toggle("active", mode === "classic");
    localStorage.setItem(KEY, mode);
  }
  const saved = localStorage.getItem(KEY) || "blue";
  apply(saved);
  if(btnBlue) btnBlue.addEventListener("click", function(){ apply("blue"); });
  if(btnClassic) btnClassic.addEventListener("click", function(){ apply("classic"); });
})();
</script>
'''

def delta_span_from_pct(x_pct: Optional[float]) -> str:
    if x_pct is None or not np.isfinite(x_pct):
        return '<span class="chg na">N/A</span>'
    if abs(x_pct) < 1e-12:
        return '<span class="chg zero">0.0%</span>'
    cls = "pos" if x_pct > 0 else "neg"
    sign = "+" if x_pct > 0 else "-"
    return f'<span class="chg {cls}">{sign}{abs(x_pct):.1f}%</span>'

def delta_span_from_usd(x: Optional[float]) -> str:
    if x is None or not np.isfinite(x):
        return '<span class="chg na">N/A</span>'
    if abs(x) < 1e-12:
        return '<span class="chg zero">$0</span>'
    cls = "pos" if x > 0 else "neg"
    sign = "+" if x > 0 else "-"
    return f'<span class="chg {cls}">{sign}{fmt_money_compact_abs(abs(x))}</span>'

def render_shared_header(page_title: str, updated_iso: str, active_tab: str = "Analysis", base_prefix: str = "") -> str:
    tabs = [
        ("Protocols", f"{base_prefix}../index.html#protocols"),
        ("Dashboard", f"{base_prefix}../index.html#dashboard"),
        ("Analysis", f"{base_prefix}index.html"),
        ("About", f"{base_prefix}../about.html"),
    ]
    tab_html = []
    for label, href in tabs:
        cls = "tab active" if label == active_tab else "tab"
        tab_html.append(f'<a class="{cls}" href="{href}">{html.escape(label)}</a>')
    return f'''
    <div class="topbar">
      <div>
        <div class="brand">
          <h1>RWA.watch</h1>
          <span class="beta">Beta Version</span>
          <div class="modeToggle" title="Delta color mode">
            <button id="modeBlue" class="modeBtn active" type="button">Blue</button>
            <button id="modeClassic" class="modeBtn" type="button">Classic</button>
          </div>
        </div>
      </div>
      <div class="updated">Updated: {html.escape(updated_iso)}</div>
    </div>
    <div class="headline">
      <h2>{html.escape(page_title)}</h2>
      <p>Daily and weekly intelligence on tokenized real-world assets. TVL trends, capital flows, market structure and monitoring signals.</p>
    </div>
    <div class="tabs">{''.join(tab_html)}</div>
    '''

def legal_footer() -> str:
    year = SNAPSHOT_DATE.year
    return f'''
    <div class="footer">
      <div class="legalLine">
        Data: DeFiLlama (public APIs) • Educational only • Not investment advice •
        <details class="legalDetails">
          <summary>Legal &amp; disclaimer</summary>
          <div class="legalBody">
            <p><strong>No investment advice.</strong> This site is provided for educational and informational purposes only and does not constitute investment, legal, or tax advice.</p>
            <p><strong>No warranty.</strong> Data is provided &quot;as is&quot; without warranties of accuracy, completeness, or timeliness. Methodologies and upstream data may change.</p>
            <p><strong>No solicitation.</strong> Nothing on this site constitutes an offer, solicitation, or recommendation to buy or sell any asset.</p>
            <p><strong>Limitation of liability.</strong> We are not liable for any losses arising from use of this site.</p>
          </div>
        </details>
      </div>
      <div class="copyright">© {year} RWA.watch — All rights reserved.</div>
    </div>
    '''



In [ ]:

# ----------------------------
# HTML RENDERING
# ----------------------------

def render_metric_card(label: str, value_html: str, sub_html: str = "", primary: bool = False, mini_html: str = "") -> str:
    # Avoid leading whitespace in grid containers (some browsers treat text nodes as grid items).
    cls = "metricCard primary" if primary else "metricCard secondary"
    mini = f'<div class="metricCardMini">{mini_html}</div>' if mini_html else ""
    return (
        f'<div class="{cls}">'
        f'<div class="metricCardLabel">{html.escape(label)}</div>'
        f'<div class="metricCardValue">{value_html}</div>'
        f'<div class="metricCardSub">{sub_html}</div>'
        f'{mini}'
        f'</div>'
    )

def render_category_bar(row: Dict[str, Any]) -> str:
    # Bar width always reflects current allocation (share of tracked TVL).
    share_pct = 0.0 if row.get("share") is None else max(0.0, float(row["share"]) * 100.0)
    d_usd = row.get("change_1d_usd")
    d_pct = row.get("change_1d_pct")

    # Once 24h history exists, make flow the primary number and keep share as context.
    if d_usd is None:
        right_primary = f"{share_pct:.1f}%"
        right_secondary = ""  # no secondary in allocation mode
        left_secondary = ""
    else:
        right_primary = delta_span_from_usd(d_usd)
        share_txt = f"Share {share_pct:.1f}%"
        pct_txt = delta_span_from_pct(d_pct)
        right_secondary = f"{share_txt} · {pct_txt}" if pct_txt != "N/A" else share_txt
        left_secondary = ""  # avoid duplicating the same delta in two places

    return f'''
    <div class="categoryBar">
      <div><b>{html.escape(str(row["category"]))}</b><div class="dim">{left_secondary}</div></div>
      <div class="barTrack"><div class="barFill" style="width:{min(100.0, share_pct):.1f}%"></div></div>
      <div style="text-align:right;"><b>{right_primary}</b><div class="dim">{right_secondary}</div></div>
    </div>
    '''

def render_rank_watch_metric(rank_watch: List[Dict[str, Any]]) -> Tuple[str, str]:
    if not rank_watch:
        return "N/A", "No prior snapshot available for Top 10 rank movement."
    moved = 0
    entered = 0
    for row in rank_watch:
        prev = row.get("previous_rank")
        rank = row.get("rank_by_tvl")
        if prev is None:
            continue
        if int(prev) > 10:
            entered += 1
        if rank is not None and int(prev) <= 10 and int(rank) != int(prev):
            moved += 1
    return f"{len(rank_watch)} assets", f"{entered} entered Top 10, {moved} moved vs yesterday."

def render_recent_recaps(recent_recaps: List[Tuple[str, str]]) -> str:
    return ''.join([f'<div class="recapItem"><a href="{html.escape(link)}">{html.escape(label)}</a></div>' for label, link in recent_recaps]) or '<div class="recapItem">No recent recaps yet.</div>'

def render_daily_page(payload: Dict[str, Any], recent_recaps: List[Tuple[str, str]], weekly_href: str = "weekly/index.html", canonical_override: Optional[str] = None) -> str:
    hm = payload["headline_metrics"]
    rank_watch_value, rank_watch_sub = render_rank_watch_metric(payload["asset_leadership"].get("rank_watch", []))
    canonical_url = canonical_override or payload.get("permalink", f"/analysis/recap-{payload['date']}.html")

    top3_mini = ''.join([
        f'<div class="miniRow"><span>{html.escape(str(d.get("name") or "N/A"))}</span><b>{fmt_pct_plain(d.get("share_pct"), 1)}</b></div>'
        for d in (hm.get("top3_details") or [])
    ])

    top10_mini = ""
    if hm.get("top4_10_share_pct") is not None:
        top10_mini = f'<div class="miniRow"><span>Ranks 4–10</span><b>{fmt_pct_plain(hm.get("top4_10_share_pct"), 1)}</b></div>'
    if hm.get("top10_concentration") is not None:
        rest_share = 100.0 - float(hm.get("top10_concentration"))
        top10_mini += f'<div class="miniRow"><span>Rest of market</span><b>{fmt_pct_plain(rest_share, 1)}</b></div>'


    gainer = hm.get("biggest_asset_gainer") or {}
    gainer_sub = delta_span_from_usd(gainer.get("change_1d_usd")) + " " + delta_span_from_pct(gainer.get("change_1d_pct"))
    if gainer.get("rank_change") is not None:
        try:
            gainer_sub += f' · Δrank {int(gainer.get("rank_change")):+d}'
        except Exception:
            pass

    near10 = hm.get("near_top10_focus") or {}
    near10_sub = "N/A"
    if near10:
        gap_usd = near10.get("gap_to_top10_usd")
        gap_pct = near10.get("gap_to_top10_pct")
        top10_name = near10.get("top10_name")
        chasing = f'Chasing: {html.escape(str(top10_name))}' if top10_name else ""
        gap_line = f'Gap to #10: {fmt_money_compact_abs(gap_usd)}'
        if gap_pct is not None:
            gap_line += f' ({fmt_pct_plain(gap_pct, 1)})'
        near10_sub = (chasing + "<br>" if chasing else "") + gap_line


    catlead = hm.get("category_leader_24h") or {}
    catlead_sub = delta_span_from_usd(catlead.get("change_1d_usd")) + " " + delta_span_from_pct(catlead.get("change_1d_pct"))

    primary_cards = [
        render_metric_card("Total RWA TVL", fmt_money_compact_abs(hm["total_rwa_tvl"]), "Aggregate tracked market value.", primary=True),
        render_metric_card("24H Change", delta_span_from_usd(hm["change_1d_usd"]), delta_span_from_pct(hm["change_1d_pct"]), primary=True),
        render_metric_card("7D Change", delta_span_from_usd(hm["change_7d_usd"]), delta_span_from_pct(hm["change_7d_pct"]), primary=True),
        render_metric_card("Top 3 Concentration", fmt_pct_plain(hm.get("top3_concentration"), 1), "Share held by the three largest tracked assets.", primary=True, mini_html=top3_mini),
    ]

    secondary_cards = [
        render_metric_card("Top 10 Concentration", fmt_pct_plain(hm.get("top10_concentration"), 1), "Share held by the ten largest tracked assets.", mini_html=top10_mini),
        render_metric_card("Category Leader (24h)", html.escape(catlead.get("category") or "N/A"), catlead_sub),
        render_metric_card("Biggest Asset Gainer (24h)", html.escape(gainer.get("name") or "N/A"), gainer_sub),
        render_metric_card("Near Top 10", html.escape(near10.get("name") or "N/A"), near10_sub),
        render_metric_card("New Entrants", str(hm.get("new_entrants", 0)), "Assets crossing the $1M threshold today."),
        render_metric_card("Assets Tracked", str(hm.get("protocols_tracked", 0)), "Tracked assets in today’s universe."),
    ]

    # Capital section copy: allocation-only until 24h deltas exist, then becomes flow-first.
    _cap_rows = payload.get("capital_allocation") or []
    _has_flow = any(r.get("change_1d_usd") is not None for r in _cap_rows)
    capital_section_title = "Where Capital Moved" if _has_flow else "Capital Allocation"
    capital_section_sub = "24h category flows (Δ$) with current share context." if _has_flow else "Category share of tracked market value."

    allocation_bars = ''.join([render_category_bar(row) for row in payload["capital_allocation"][:6]]) or '<div class="rowCard">No category allocation data available.</div>'

    largest_assets_rows = []
    for row in payload["market_structure"]["largest_assets"][:8]:
        largest_assets_rows.append(f'''
        <tr>
          <td>{html.escape(row["name"])}</td>
          <td>{html.escape(str(row.get("analytics_category") or "Other RWA"))}</td>
          <td>{fmt_money_compact_abs(row["tvl_usd"])}</td>
          <td>{f"{row['share']*100:.1f}%" if row.get('share') is not None else "N/A"}</td>
        </tr>
        ''')

    increase_rows, decline_rows, rank_watch_rows = [], [], []

    for row in payload["asset_leadership"].get("largest_increase", [])[:4]:
        increase_rows.append(
            f'<div class="rowCard"><div class="rowTitle"><b>{html.escape(row["name"])}</b><span>{delta_span_from_usd(row.get("change_1d_usd"))}</span></div><div class="rowMeta">{fmt_money_compact_abs(row.get("tvl_usd"))} · {html.escape(str(row.get("analytics_category") or "Other RWA"))}</div></div>'
        )

    for row in payload["asset_leadership"].get("largest_decline", [])[:4]:
        change = row.get("change_1d_usd")
        if change is None or change >= 0:
            continue
        decline_rows.append(
            f'<div class="rowCard"><div class="rowTitle"><b>{html.escape(row["name"])}</b><span>{delta_span_from_usd(change)}</span></div><div class="rowMeta">{fmt_money_compact_abs(row.get("tvl_usd"))} · {html.escape(str(row.get("analytics_category") or "Other RWA"))}</div></div>'
        )

    for row in payload["asset_leadership"].get("rank_watch", [])[:10]:
        prev_rank = row.get("previous_rank")
        if prev_rank in (None, 999999):
            status = "New to tracked rank watch"
        elif row.get("rank_by_tvl") and prev_rank:
            if row["rank_by_tvl"] < prev_rank:
                status = f"Moved up from #{int(prev_rank)}"
            elif row["rank_by_tvl"] > prev_rank:
                status = f"Moved down from #{int(prev_rank)}"
            else:
                status = f"Held rank #{int(prev_rank)}"
        else:
            status = row.get("status") or "Rank watch"
        rank_watch_rows.append(
            f'<div class="rowCard"><div class="rowTitle"><b>{html.escape(row["name"])}</b><span>Rank #{int(row["rank_by_tvl"])}</span></div><div class="rowMeta">{html.escape(status)}</div></div>'
        )

    if not increase_rows:
        increase_rows.append('<div class="rowCard">No validated increase signal available yet.</div>')
    if not decline_rows:
        decline_rows.append('<div class="rowCard">No validated daily declines available yet.</div>')
    if not rank_watch_rows:
        rank_watch_rows.append('<div class="rowCard">No validated Top 10 rank pressure signal yet.</div>')

    category_trends_html = []
    for row in payload["category_trends"][:6]:
        # Trends panel focuses on flows (changes), while Allocation panel shows the current share.
        d1 = delta_span_from_usd(row.get("change_1d_usd"))
        d7 = delta_span_from_usd(row.get("change_7d_usd"))
        share_pct = None if row.get('share') is None else float(row.get('share'))*100.0
        share_txt = fmt_pct_plain(share_pct, 1)
        category_trends_html.append(
            f'<div class="rowCard">'
            f'  <div class="rowTitle"><b>{html.escape(str(row["category"]))} {row["arrow"]}</b><span>{d1}</span></div>'
            f'  <div class="rowMeta">7D {d7} · Share {share_txt}</div>'
            f'</div>'
        )

    signals = payload.get("monitoring_signals", [])[:3]
    watch_items = payload.get("what_to_watch_tomorrow", [])[:3]
    signals_html = ''.join([f'<div class="signalItem">{html.escape(item)}</div>' for item in signals]) or '<div class="signalItem">No monitoring signals were generated for this snapshot.</div>'
    watch_html = ''.join([f'<div class="watchItem">{html.escape(item)}</div>' for item in watch_items]) or '<div class="watchItem">The watchlist will expand as local history accumulates.</div>'
    recent_html = render_recent_recaps(recent_recaps)

    header = render_shared_header("Real-World Asset Intelligence & Monitoring", iso_ts(RUN_TS_UTC), active_tab="Analysis", base_prefix="")
    footer = legal_footer()

    return f'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<title>{html.escape(payload["title"])} — RWA.watch</title>
<meta name="description" content="{html.escape(payload["summary"][:220])}" />
<link rel="canonical" href="{html.escape(canonical_url)}" />
<meta property="og:title" content="{html.escape(payload["title"])} — RWA.watch" />
<meta property="og:description" content="{html.escape(payload["summary"][:220])}" />
<meta property="og:image" content="{html.escape(payload["main_chart"]["image"])}" />
<meta property="og:url" content="{html.escape(canonical_url)}" />
<meta name="twitter:card" content="summary_large_image" />
<style>{SHELL_CSS}</style>
<style>{ANALYTICS_CSS}</style>
</head>
<body>
<div class="container">
  {header}

  <section class="heroShell">
    <div class="heroEyebrow">Daily analysis</div>
    <div class="heroTop">
      <div class="heroLeft">
        <div>
          <h1 class="heroTitle">{html.escape(payload["title"])}</h1>
          <div class="heroDate">{html.escape(payload["display_date"])} (UTC)</div>
          <p class="heroSummary">{html.escape(payload["summary"])}</p>
          <div class="kpiStrip">
            <span class="badgePill">Top 3 Assets: {html.escape(", ".join(hm["top3_assets"] or []))}</span>
            <span class="badgePill">Top 10 Concentration: {fmt_pct_plain(hm.get("top10_concentration"),1)}</span>
          </div>
          <div class="quickNav">
            <a href="index.html">Latest</a>
            <a href="{html.escape(weekly_href)}">Weekly</a>
            <a href="#recent-analysis">Last 7 Days</a>
          </div>
        </div>
      </div>
      <div class="heroChartWrap">
        <div class="heroChartCaption">{html.escape(payload["main_chart"]["title"])}</div>
        <img src="{html.escape(payload["main_chart"]["image"])}" alt="{html.escape(payload["main_chart"]["title"])}" />
        <div class="heroChartLabel">{fmt_money_compact_abs(hm["total_rwa_tvl"])}</div>
      </div>
    </div>
    <div class="marketRegime"><b>Market Regime:</b> {html.escape(payload.get("market_regime") or "N/A")}</div>
  </section>

  <section class="metricGridPrimary">{"".join(primary_cards).strip()}</section>

  <section class="metricGridSecondary">{"".join(secondary_cards).strip()}</section>

  <div class="sectionGrid">
    <section class="sectionCard">
      <h3>{html.escape(capital_section_title)}</h3>
      <p class="sub">{html.escape(capital_section_sub)}</p>
      {allocation_bars}
    </section>

    <section class="sectionCard">
      <h3>Market Structure</h3>
      <p class="sub">Current concentration and the assets shaping the market-wide picture.</p>
      <div class="badgeWrap">
        <span class="badgePill">Top 3 Concentration: {fmt_pct_plain(payload["market_structure"]["top3_concentration_pct"], 1)}</span>
        <span class="badgePill">Top 10 Concentration: {fmt_pct_plain(payload["market_structure"]["top10_concentration_pct"], 1)}</span>
      </div>
      <div class="previewBox">Top 3 assets: {html.escape(", ".join(payload["market_structure"]["top3_assets"]))}</div>
      <table class="miniTable">
        <thead><tr><th>Largest Assets</th><th>Category</th><th>TVL</th><th>Share</th></tr></thead>
        <tbody>{''.join(largest_assets_rows)}</tbody>
      </table>
    </section>
  </div>

  <div class="sectionGrid equal">
    <section class="sectionCard">
      <h3>Asset Leadership</h3>
      <p class="sub">Largest validated moves by tracked value, plus rank pressure around the Top 10.</p>
      <div class="dualPanels">
        <div class="miniPanel">
          <h4>Largest TVL Increase</h4>
          <div class="inlineList">{''.join(increase_rows)}</div>
        </div>
        <div class="miniPanel">
          <h4>Largest TVL Decline</h4>
          <div class="inlineList">{''.join(decline_rows)}</div>
        </div>
      </div>
      <div class="miniPanel" style="margin-top:12px;">
        <h4>Top 10 Rank Watch</h4>
        <div class="inlineList">{''.join(rank_watch_rows)}</div>
      </div>
    </section>

    <section class="sectionCard">
      <h3>Category Trends</h3>
      <p class="sub">Directional view across the main investor-facing RWA categories.</p>
      <div class="inlineList">{''.join(category_trends_html) or '<div class="rowCard">No category trend data available.</div>'}</div>
    </section>
  </div>

  <div class="sectionGrid equal">
    <section class="sectionCard">
      <h3>Monitoring Signals</h3>
      <p class="sub">Validated factual observations from the current snapshot and local history.</p>
      <div class="signalList">{signals_html}</div>
    </section>

    <section class="sectionCard">
      <h3>What to Watch Tomorrow</h3>
      <p class="sub">Short factual watchpoints for the next UTC snapshot.</p>
      <div class="watchList">{watch_html}</div>
    </section>
  </div>

  <div class="recentRow" id="recent-analysis">
    <section class="sectionCard">
      <h3>Share Recap</h3>
      <p class="sub">Each daily note is designed to be linkable and shareable on X and elsewhere.</p>
      <div class="shareActions">
        <a class="shareBtn" href="{html.escape(canonical_url)}">Copy link</a>
        <a class="shareBtn" href="https://x.com/intent/post?text={requests.utils.quote(payload['x_share_text'])}" target="_blank" rel="noopener">Copy X text</a>
        <a class="shareBtn" href="{html.escape(payload['main_chart']['image'])}" target="_blank" rel="noopener">Share chart</a>
      </div>
      <div class="copyBox">{html.escape(payload["x_share_text"])}</div>
    </section>

    <section class="sectionCard">
      <h3>Recent Analysis</h3>
      <p class="sub">Only the latest seven daily recaps are surfaced in the on-site navigation.</p>
      <div class="recapList">{recent_html}</div>
      <div class="archiveNote">Older recap URLs remain retained and indexable, but they are not exposed as a full on-site archive list.</div>
      <div class="recapList" style="margin-top:12px;"><div class="recapItem"><a href="{html.escape(weekly_href)}">Open weekly recap</a></div></div>
    </section>
  </div>

  {footer}
</div>
{TOGGLE_SCRIPT}
</body>
</html>
'''

def render_weekly_page(payload: Dict[str, Any]) -> str:
    header = render_shared_header("Real-World Asset Intelligence & Monitoring", iso_ts(RUN_TS_UTC), active_tab="Analysis", base_prefix="../")
    footer = legal_footer()

    hm = payload.get("headline_metrics", {})
    week_primary = [
        render_metric_card("Total RWA TVL", fmt_money_compact_abs(hm.get("total_rwa_tvl")), "Aggregate tracked market value.", primary=True),
        render_metric_card("7D Change", delta_span_from_usd(hm.get("change_7d_usd")), delta_span_from_pct(hm.get("change_7d_pct")), primary=True),
        render_metric_card("Top 3 Concentration", fmt_pct_plain(payload["concentration_change"].get("top3_concentration_pct"), 1), "Share held by the three largest assets.", primary=True),
        render_metric_card("Top 10 Concentration", fmt_pct_plain(payload["concentration_change"].get("top10_concentration_pct"), 1), "Share held by the ten largest assets.", primary=True),
    ]

    shift_cards = []
    for row in payload.get("category_shifts", [])[:6]:
        shift_cards.append(
            f'<div class="rowCard"><div class="rowTitle"><b>{html.escape(str(row["category"]))}</b><span>{fmt_pct_plain((None if row.get('share') is None else float(row.get('share'))*100.0), 1)}</span></div><div class="rowMeta">{delta_span_from_usd(row.get("change_7d_usd"))} · {delta_span_from_pct(row.get("change_7d_pct"))}</div></div>'
        )
    if not shift_cards:
        shift_cards.append('<div class="rowCard">Weekly category history will populate as UTC snapshots accumulate.</div>')

    latest_assets_rows = []
    latest_assets = payload.get("largest_asset_change", {}).get("largest_increase", [])
    for row in latest_assets[:8]:
        latest_assets_rows.append(f'''
        <tr>
          <td>{html.escape(row.get("name",""))}</td>
          <td>{html.escape(str(row.get("analytics_category") or "Other RWA"))}</td>
          <td>{fmt_money_compact_abs(row.get("tvl_usd"))}</td>
          <td>{delta_span_from_usd(row.get("change_1d_usd"))}</td>
        </tr>
        ''')
    if not latest_assets_rows:
        latest_assets_rows.append('<tr><td colspan="4">Weekly top-asset detail will populate as local history expands.</td></tr>')

    entrants = payload.get("new_entrants", [])
    entrants_html = "".join(
        [f'<div class="rowCard"><div class="rowTitle"><b>{html.escape(row["name"])}</b><span>Rank #{int(row["rank_by_tvl"])}</span></div><div class="rowMeta">{fmt_money_compact_abs(row["tvl_usd"])} · {html.escape(str(row.get("analytics_category") or "Other RWA"))}</div></div>' for row in entrants[:6]]
    ) or '<div class="rowCard">No validated weekly entrants available yet.</div>'

    watch_html = ''.join([f'<div class="watchItem">{html.escape(item)}</div>' for item in payload.get("what_to_watch_next_week", [])[:3]]) or '<div class="watchItem">Next-week watchpoints will populate automatically as more local history is collected.</div>'

    return f'''<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<title>{html.escape(payload["title"])} — RWA.watch</title>
<meta name="description" content="Weekly RWA market recap covering category shifts, concentration and rank pressure." />
<link rel="canonical" href="/analysis/weekly/weekly-{html.escape(payload['week_label'])}.html" />
<style>{SHELL_CSS}</style>
<style>{ANALYTICS_CSS}</style>
</head>
<body>
<div class="container">
  {header}

  <section class="heroShell">
    <div class="heroEyebrow">Weekly recap</div>
    <h1 class="heroTitle">{html.escape(payload["title"])}</h1>
    <div class="heroDate">{html.escape(payload["display_week"])} (UTC)</div>
    <p class="heroSummary">{html.escape(payload["summary"])}</p>
    <div class="quickNav">
      <a href="../index.html">Latest</a>
      <a href="weekly-{html.escape(payload['week_label'])}.html">Weekly</a>
      <a href="../index.html#recent-analysis">Last 7 Days</a>
    </div>
  </section>

  <section class="metricGridPrimary">
    {''.join(week_primary)}
  </section>

  <div class="sectionGrid equal">
    <section class="sectionCard">
      <h3>Category Shifts</h3>
      <p class="sub">Investor-facing category changes based on the latest available weekly comparison window.</p>
      <div class="inlineList">{''.join(shift_cards)}</div>
    </section>

    <section class="sectionCard">
      <h3>What to Watch Next Week</h3>
      <p class="sub">Concise factual watchpoints for the next weekly rollup.</p>
      <div class="watchList">{watch_html}</div>
    </section>
  </div>

  <div class="sectionGrid equal">
    <section class="sectionCard">
      <h3>Largest Asset Changes</h3>
      <p class="sub">Assets contributing the most visible change into the current weekly window.</p>
      <table class="miniTable">
        <thead><tr><th>Asset</th><th>Category</th><th>TVL</th><th>Change</th></tr></thead>
        <tbody>{''.join(latest_assets_rows)}</tbody>
      </table>
    </section>

    <section class="sectionCard">
      <h3>New Entrants</h3>
      <p class="sub">Assets crossing the tracked threshold in the current weekly window.</p>
      <div class="inlineList">{entrants_html}</div>
    </section>
  </div>

  {footer}
</div>
{TOGGLE_SCRIPT}
</body>
</html>
'''

def render_analysis_index(current_payload: Dict[str, Any], recent_recaps: List[Tuple[str, str]], weekly_links: List[Tuple[str, str]]) -> str:
    return render_daily_page(current_payload, recent_recaps, weekly_href=weekly_links[0][1] if weekly_links else "weekly/index.html", canonical_override=current_payload.get("permalink"))


In [ ]:

# ----------------------------
# RECENT LISTS + PAGE WRITE
# ----------------------------

def recap_link_for_date(d: str) -> str:
    return f"recap-{d}.html"

all_dates_sorted = sorted(all_snapshots.keys(), reverse=True)
recent_dates = [d for d in all_dates_sorted if (SNAPSHOT_DATE - date.fromisoformat(d)).days <= 7]
recent_recaps = [(f"{date_to_human(date.fromisoformat(d))} — Daily recap", recap_link_for_date(d)) for d in recent_dates]
weekly_links = [(f"Week {ISO_WEEK}, {ISO_YEAR}", f"weekly/weekly-{WEEK_LABEL}.html")]
weekly_href = weekly_links[0][1]

daily_html = render_daily_page(daily_page_payload, recent_recaps, weekly_href=weekly_href)
daily_path = SITE_ANALYSIS_ROOT / f"recap-{SNAPSHOT_DATE_STR}.html"
daily_path.write_text(daily_html, encoding="utf-8")

# Analysis opens directly to the latest daily recap.
index_html = render_daily_page(daily_page_payload, recent_recaps, weekly_href=weekly_href, canonical_override=daily_page_payload.get("permalink"))
index_path = SITE_ANALYSIS_ROOT / "index.html"
index_path.write_text(index_html, encoding="utf-8")

weekly_html = render_weekly_page(weekly_payload)
weekly_path = SITE_WEEKLY_ROOT / f"weekly-{WEEK_LABEL}.html"
weekly_path.write_text(weekly_html, encoding="utf-8")

print("Rendered:")
print("-", index_path)
print("-", daily_path)
print("-", weekly_path)

# Lightweight sitemap: keep historical recap URLs indexable without exposing a full archive list in the UI.
BASE_URL = PUBLIC_BASE_URL.rstrip("/") if "PUBLIC_BASE_URL" in globals() and PUBLIC_BASE_URL else "https://rwa.watch/analysis"
sitemap_urls = [f"{BASE_URL}/index.html", f"{BASE_URL}/recap-{SNAPSHOT_DATE_STR}.html", f"{BASE_URL}/weekly/weekly-{WEEK_LABEL}.html"]
for d in all_dates_sorted:
    sitemap_urls.append(f"{BASE_URL}/recap-{d}.html")

sitemap_xml = ['<?xml version="1.0" encoding="UTF-8"?>', '<urlset xmlns="http://www.sitemaps.org/schemas/sitemap/0.9">']
for u in dict.fromkeys(sitemap_urls):
    sitemap_xml.append(f"  <url><loc>{html.escape(u)}</loc></url>")
sitemap_xml.append('</urlset>')
sitemap_path = SITE_ANALYSIS_ROOT / "sitemap.xml"
sitemap_path.write_text("\n".join(sitemap_xml), encoding="utf-8")
print("-", sitemap_path)


In [ ]:
# ----------------------------
# ZIP PACKAGE
# ----------------------------

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    # Always include the site itself.
    for path in SITE_ROOT.rglob("*"):
        if path.is_file():
            zf.write(path, path.relative_to(ROOT_DIR))

    # Include private snapshot/validation/articles/charts and optionally exports for internal use.
    private_paths = [SNAPSHOT_ROOT, VALIDATION_ROOT, ARTICLE_ROOT, CHARTS_ROOT, NORMALIZED_ROOT, (PRIVATE_DATA_ROOT / "sources")]
    if INCLUDE_PRIVATE_EXPORTS_IN_ZIP:
        private_paths.append(EXPORTS_ROOT)

    for root in private_paths:
        for path in root.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(ROOT_DIR))

print("ZIP ready:", ZIP_PATH)


In [ ]:
# ----------------------------
# QUICK BUILD SUMMARY
# ----------------------------

summary = {
    "snapshot_date": SNAPSHOT_DATE_STR,
    "site_index": str(index_path),
    "daily_page": str(daily_path),
    "weekly_page": str(weekly_path),
    "snapshot_json": str(snapshot_path),
    "validation_json": str(validation_path),
    "market_metrics_csv": str(market_metrics_path),
    "category_timeseries_csv": str(category_timeseries_path),
    "asset_timeseries_csv": str(asset_timeseries_path),
    "zip_path": str(ZIP_PATH),
    "sitemap_xml": str(sitemap_path),
}
print(json.dumps(summary, indent=2))

try:
    from google.colab import files
    print("\nColab detected. Triggering ZIP download...")
    files.download(str(ZIP_PATH))
except Exception as exc:
    print("\nAutomatic download skipped (not running inside Colab or download hook unavailable).")
    print("Manual file:", ZIP_PATH)
